# cli

> Ramabana in a terminal: a transcript of blocks, a status bar, and one line to type in.

Built on [teleprint](https://github.com/answerdotai/teleprint), whose centre is the same as this app's: an append-mostly transcript whose durable rendering is the terminal's own scrollback. Tool calls are *blocks* rather than lines, which is what makes them foldable. The answer stays readable and the thirty tool results it took are one click away.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import asyncio, tempfile, threading
from fastcore.test import test_eq, test_fail, test_ne
from teleprint.keys import Key
from teleprint.testing import EmuTty
from ramabana.core import resolve
from ramabana.testing import FullHost, fake_agent

In [ ]:
#| export
import asyncio, difflib, os, re, shlex, shutil, subprocess, sys, tempfile, threading, time
from base64 import b64encode
from dataclasses import dataclass
from importlib.util import find_spec
from pathlib import Path
from urllib.parse import unquote, urlparse
from rich.text import Text
from rich.markdown import Markdown
from rich.cells import cell_len
from rich.theme import Theme
from fastcore.script import call_parse
from fastcore.basics import patch
from teleprint.buffer import Buffer
from teleprint.compositor import Compositor
from teleprint.transcript import TranscriptView
from teleprint.tty import RealTty
from teleprint.widgets import CompletionMenu, Tooltip
from ramabana.core import accepts, agent_err, env
from ramabana.tools import WRITE_TOOLS, LocalHost, media_dir, save_media
from ramabana.agent import Agent, Approvals, answer_md
from datetime import datetime
from ramabana import __version__

## Blocks and keys

Eight kinds of block, one gutter each. A turn is a timeline of them. A `step` for each stretch of narration and a `tool` for each call it introduced, in the order the work actually happened, then one `reply` at the end holding the answer. Steps and results fold to a single row, so a turn of thirty calls reads as thirty lines, and the answer is never underneath the narration that led to it.

In [ ]:
#| export
DARK = {
    'bg0': '#111210', 'bg1': '#191b18', 'bg2': '#30332d',
    'fg0': '#f2f0e8', 'fg1': '#d6d3c8', 'gray': '#7c8077',
    'red': '#c97970', 'green': '#91aa8f', 'yellow': '#c3a66f',
    'blue': '#9bb59a', 'aqua': '#b8c7b5', 'orange': '#b49a7b',
}
LIGHT = {
    'bg0': '#f8f5ec', 'bg1': '#eee9dc', 'bg2': '#d8d1c1',
    'fg0': '#262924', 'fg1': '#454940', 'gray': '#77776d',
    'red': '#9a443b', 'green': '#3f6d4d', 'yellow': '#87611c',
    'blue': '#246275', 'aqua': '#3e6660', 'orange': '#8a5b2b',
}
THEMES = {'dark': DARK, 'light': LIGHT}

#: The rest are the schemes a Ghostty-family terminal -- conterm, Ghostty itself -- already ships,
#: mapped onto the twelve semantic keys above rather than onto ANSI slots. Set the terminal's own
#: theme to the scheme of the same name and the surface stops fighting the window around it.
THEMES.update({
    'gruvbox': {
        'bg0': '#282828', 'bg1': '#3c3836', 'bg2': '#504945',
        'fg0': '#fbf1c7', 'fg1': '#ebdbb2', 'gray': '#928374',
        'red': '#fb4934', 'green': '#b8bb26', 'yellow': '#fabd2f',
        'blue': '#83a598', 'aqua': '#8ec07c', 'orange': '#fe8019',
    },
    'gruvbox-light': {
        'bg0': '#fbf1c7', 'bg1': '#ebdbb2', 'bg2': '#d5c4a1',
        'fg0': '#282828', 'fg1': '#3c3836', 'gray': '#7c6f64',
        'red': '#9d0006', 'green': '#79740e', 'yellow': '#b57614',
        'blue': '#076678', 'aqua': '#427b58', 'orange': '#af3a03',
    },
    'nord': {
        'bg0': '#2e3440', 'bg1': '#3b4252', 'bg2': '#434c5e',
        'fg0': '#eceff4', 'fg1': '#d8dee9', 'gray': '#616e88',
        'red': '#bf616a', 'green': '#a3be8c', 'yellow': '#ebcb8b',
        'blue': '#81a1c1', 'aqua': '#88c0d0', 'orange': '#d08770',
    },
    'tokyonight': {
        'bg0': '#1a1b26', 'bg1': '#24283b', 'bg2': '#414868',
        'fg0': '#c0caf5', 'fg1': '#a9b1d6', 'gray': '#565f89',
        'red': '#f7768e', 'green': '#9ece6a', 'yellow': '#e0af68',
        'blue': '#7aa2f7', 'aqua': '#7dcfff', 'orange': '#ff9e64',
    },
    'catppuccin': {
        'bg0': '#1e1e2e', 'bg1': '#313244', 'bg2': '#45475a',
        'fg0': '#cdd6f4', 'fg1': '#bac2de', 'gray': '#7f849c',
        'red': '#f38ba8', 'green': '#a6e3a1', 'yellow': '#f9e2af',
        'blue': '#89b4fa', 'aqua': '#94e2d5', 'orange': '#fab387',
    },
    'latte': {
        'bg0': '#eff1f5', 'bg1': '#e6e9ef', 'bg2': '#ccd0da',
        'fg0': '#4c4f69', 'fg1': '#5c5f77', 'gray': '#8c8fa1',
        'red': '#d20f39', 'green': '#40a02b', 'yellow': '#df8e1d',
        'blue': '#1e66f5', 'aqua': '#179299', 'orange': '#fe640b',
    },
    'everforest': {
        'bg0': '#2d353b', 'bg1': '#343f44', 'bg2': '#3d484d',
        'fg0': '#d3c6aa', 'fg1': '#9da9a0', 'gray': '#859289',
        'red': '#e67e80', 'green': '#a7c080', 'yellow': '#dbbc7f',
        'blue': '#7fbbb3', 'aqua': '#83c092', 'orange': '#e69875',
    },
    'dracula': {
        'bg0': '#282a36', 'bg1': '#343746', 'bg2': '#44475a',
        'fg0': '#f8f8f2', 'fg1': '#e2e2dc', 'gray': '#6272a4',
        'red': '#ff5555', 'green': '#50fa7b', 'yellow': '#f1fa8c',
        'blue': '#bd93f9', 'aqua': '#8be9fd', 'orange': '#ffb86c',
    },
    'kanagawa': {
        'bg0': '#1f1f28', 'bg1': '#2a2a37', 'bg2': '#363646',
        'fg0': '#dcd7ba', 'fg1': '#c8c093', 'gray': '#727169',
        'red': '#e46876', 'green': '#98bb6c', 'yellow': '#e6c384',
        'blue': '#7e9cd8', 'aqua': '#7aa89f', 'orange': '#ffa066',
    },
    'solarized': {
        'bg0': '#002b36', 'bg1': '#073642', 'bg2': '#0f4b5a',
        'fg0': '#fdf6e3', 'fg1': '#93a1a1', 'gray': '#657b83',
        'red': '#dc322f', 'green': '#859900', 'yellow': '#b58900',
        'blue': '#268bd2', 'aqua': '#2aa198', 'orange': '#cb4b16',
    },
    'solarized-light': {
        'bg0': '#fdf6e3', 'bg1': '#eee8d5', 'bg2': '#d9d2bd',
        'fg0': '#002b36', 'fg1': '#586e75', 'gray': '#93a1a1',
        'red': '#dc322f', 'green': '#859900', 'yellow': '#b58900',
        'blue': '#268bd2', 'aqua': '#2aa198', 'orange': '#cb4b16',
    },
})

#: The pale ones, which want a pale code theme behind their Markdown as well.
PALE_THEMES = frozenset({'light', 'gruvbox-light', 'latte', 'solarized-light'})

#: A pygments style per palette, for the code inside a reply. Only styles pygments really ships.
CODE_THEMES = {'gruvbox': 'gruvbox-dark', 'gruvbox-light': 'gruvbox-light', 'nord': 'nord',
               'dracula': 'dracula', 'solarized': 'solarized-dark',
               'solarized-light': 'solarized-light', 'tokyonight': 'one-dark',
               'catppuccin': 'one-dark', 'kanagawa': 'zenburn', 'everforest': 'gruvbox-dark',
               'latte': 'friendly', 'light': 'friendly'}

KAKU = DARK
GRUVBOX = KAKU  # compatibility name for extensions. Updated by set_theme
ACTIVE_THEME = 'dark'


def code_theme(name=None):
    "The pygments style to highlight code with under the named palette."
    return CODE_THEMES.get(name or ACTIVE_THEME, 'github-dark')


def set_theme(name='dark'):
    "Select the active semantic palette. `auto` safely falls back to dark in a terminal."
    global ACTIVE_THEME, GRUVBOX, MARKDOWN_THEME, GUTTERS
    name = str(name or 'dark').lower()
    if name == 'auto': name = 'dark'
    if name not in THEMES:
        near = difflib.get_close_matches(name, THEMES, n=2, cutoff=0.6)
        hint = (f'did you mean {" or ".join(repr(n) for n in near)}?' if near
                else f'/theme lists all {len(THEMES)} of them')
        raise ValueError(f'unknown theme {name!r}; {hint}')
    GRUVBOX = THEMES[name]
    ACTIVE_THEME = name
    if '_theme_parts' in globals(): MARKDOWN_THEME, GUTTERS = _theme_parts(GRUVBOX)
    return name


def _theme_parts(palette):
    markdown = Theme({
        'markdown.h1': f"bold {palette['blue']}", 'markdown.h2': f"bold {palette['yellow']}",
        'markdown.h3': f"bold {palette['aqua']}", 'markdown.h4': f"bold {palette['blue']}",
        'markdown.h5': f"bold {palette['blue']}", 'markdown.h6': palette['gray'],
        'markdown.code': palette['aqua'], 'markdown.link': f"underline {palette['blue']}",
        'markdown.link_url': palette['gray'], 'markdown.block_quote': palette['gray'],
        'markdown.item.bullet': f"bold {palette['blue']}",
        'markdown.item.number': f"bold {palette['blue']}", 'markdown.hr': palette['bg2'],
    })
    gutters = {
        'user': (Text('▌ ', style=f"bold {palette['blue']}"), Text('  ')),
        'banner': (Text('  '), Text('  ')),
        'reply': (Text('  '), Text('  ')), 'tool': (Text('│ ', style=palette['blue']), Text('  ')),
        'ask': (Text('◆ ', style=f"bold {palette['yellow']}"), Text('  ')),
        'plan': (Text('▸ ', style=f"bold {palette['yellow']}"), Text('  ')),
        'note': (Text('· ', style=palette['gray']), Text('  ')),
        'step': (Text('┆ ', style=palette['gray']), Text('  ')),
        'error': (Text('× ', style=f"bold {palette['red']}"), Text('  ')),
    }
    return markdown, gutters

MARKDOWN_THEME, GUTTERS = _theme_parts(GRUVBOX)
FOLD, FOLD_TOOL, NOTIFY_EVERY = 12, 1, 0.08

FOLD_RUNNING = True
ACT_EVERY = 0.05
FOLD_STEP = 1
STREAM_EVERY = 0.05
ACT_TAIL = 3
MAX_GROUP_ROWS = 8
MOUSE_ON, MOUSE_OFF = '\x1b[?1000;1006h', '\x1b[?1000;1006l'
SURFACE_COMMANDS = ('agent', 'attach', 'copy', 'detach', 'exit', 'guide', 'help', 'join', 'kernels',
                    'mouse', 'paste', 'promote', 'python', 'quit', 'root', 'theme', 'vars')

HELP = """normal  enter send · tab complete /commands · ctrl+t plan · ctrl+p/n history · ↑/↓ or ctrl+r transcript · ctrl+o fold the working · alt+1..9 drill in · ctrl+c stop · ctrl+d quit
timeline  a turn reads top to bottom · ┆ narration · │ a call · the answer last · ctrl+o all the working · alt+1..9 one entry
transcript  ↑/↓ blocks · pgup/pgdn page · /? search · n/N matches · g/G ends · y copy block · i compose · esc leave
edit    ctrl+a/e ends · ctrl+u/k cut line · ctrl+w cut word · ctrl+y yank
media   drop or paste a path to attach · @path in a prompt · /attach PATH · /detach [N] · ctrl+v or /paste clipboard image
copy    select with the mouse as in any scrollback · /copy the last reply · /copy turn for all of it · ctrl+r then y for any block
approve y approve · n refuse · a approve all · ctrl+y approve with a note · or type a reason and press enter to refuse
options ↑/↓ move · enter choose · an option's own letter picks it · esc cancel and keep the line
python  /python takes the line · /agent hands it back · enter runs what compiles · tab completes names · ctrl+c interrupts the cell · /vars · /promote NAME
plan    /plan · /todo TEXT · /todo ID done|active|pending|cancel · ctrl+t show/hide · survives stop and /resume
extra   /root · /root add PATH to open another folder · /theme [NAME|next] · /mouse to click blocks on the main screen · /tool-budget · /steps · /models · /model NAME · /sessions · /resume [ID|latest] · /cost · /compact · /reload
subagent /subagents shows whether delegated work may write · /subagents on|off changes it for this session
api     start with --spec · then api_load URL-or-path · api_ops · api_call"""


#: `[ ]` pending, `[▸]` active, `[x]` done, `[-]` cancelled -- see `agent.TODO_MARK`
_TODO_RE = re.compile(r'^(\[[ x▸\-]\])\s+(`[^`]*`)?\s*(.*)$')

def plan_text(md):
    """The plan checklist as themed `Text`: one colour for the marks, another for the step text.

    Read as a column of marks first and prose second, so the id and any note recede to gray rather
    than competing with the step. `GRUVBOX` is read per call, so `/theme` restyles a painted plan.
    """
    out = Text()
    for i, line in enumerate(md.splitlines()):
        if i: out.append('\n')
        m = _TODO_RE.match(line)
        if m is None:                                  # the `**title**  ·  n/m done` header
            title, sep, count = line.replace('**', '').partition('  ·  ')
            out.append(title, style=f"bold {GRUVBOX['yellow']}")
            if sep: out.append(f'  ·  {count}', style=GRUVBOX['gray'])
            continue
        mark, tid, rest = m.group(1), m.group(2) or '', m.group(3)
        out.append(mark, style=f"bold {GRUVBOX['aqua']}")
        if tid: out.append(' ' + tid.strip('`'), style=GRUVBOX['gray'])
        step, sep, note = rest.partition('  -- ')
        out.append(' ' + step, style=GRUVBOX['fg1'])
        if sep: out.append('  -- ' + note, style=GRUVBOX['gray'])
    return out


## The emblem

The terminal says two things about itself without being asked: which palette it is wearing, and
whether it is working. Both are the same drawing. `banner` opens the session with Rama's bow drawn
and the arrow on the string, aimed at the name; `arrow_mark` keeps that arrow in the status bar,
nocked and green while the session waits, loosed and flying while a turn runs, and the string slack
and gray when no model loaded at all. `flight_line` gives the running turn a row of its own.

None of it holds a colour. Every one of these reads `GRUVBOX` at the moment it is called, the way
`plan_text` does, so `/theme` restyles the emblem along with everything else already on screen.


In [ ]:
#| export
#: Rama's bow, drawn, with the arrow still on the string. Two rules hold it together.
#:
#: Every stroke really joins the next. A terminal that draws its own box characters -- Ghostty, and
#: so conterm -- makes them meet exactly across a cell edge, which means it also shows up every
#: place they do not: the first draft leant on diagonals and rounded corners that a *font* blurs
#: into looking joined, and rendered honestly it came out a bag of loose ends. So the string is one
#: straight stave and the limb is a staircase of arcs, each row entering at the column the row
#: above left at. The lower half is the upper one mirrored, `╭`for`╰` and `╮`for`╯`, so the two
#: cannot drift apart. Change a row and check the column it hands on, or the bow comes apart.
#:
#: And the bow is light strokes while the arrow is heavy ones (`≺ ━ ▸`), the two sets disjoint, so
#: `_glyphs` can colour them apart without anyone having to say where one ends.
BOW = ('     ╭───╮',
       '     │   ╰───╮',
       '     │       ╰──╮',
       '     │          ╰─╮',
       '     │            ╰╮',
       ' ≺━━━┿━━━━━━━━━━━━━┿━━━━━━━▸',
       '     │            ╭╯',
       '     │          ╭─╯',
       '     │       ╭──╯',
       '     │   ╭───╯',
       '     ╰───╯')

#: RAMABANA in double-ruled box drawing, and in glyphs the bow never uses.
WORDMARK = ('╦═╗╔═╗╔╦╗╔═╗╔╗ ╔═╗╔╗╔╔═╗',
            '╠╦╝╠═╣║║║╠═╣╠╩╗╠═╣║║║╠═╣',
            '╩╚═╩ ╩╩ ╩╩ ╩╚═╝╩ ╩╝╚╝╩ ╩')

_ARROW_GLYPHS, _BOW_GLYPHS = '≺━▸', '╭╮╰╯─│┿'
BANNER_GAP = 3
#: The width the full banner needs. Below it `banner` drops to the wordmark, and then to one line.
BANNER_MIN = max(map(len, BOW)) + BANNER_GAP + len(WORDMARK[0])


def _spans(cells):
    "`(glyph, style)` pairs as one `Text`, a span per run that styles the same."
    out, run, cur = Text(), '', None
    for ch, style in cells:
        if style != cur and run: out.append(run, style=cur); run = ''
        cur, run = style, run + ch
    if run: out.append(run, style=cur)
    return out


def _glyphs(row, palette):
    "One row of the emblem, coloured by glyph: the arrow warm, the bow quiet behind it."
    def style(ch):
        if ch == '▸': return f"bold {palette['yellow']}"
        if ch in _ARROW_GLYPHS: return palette['orange']
        return palette['gray'] if ch in _BOW_GLYPHS else None
    return _spans((ch, style(ch)) for ch in row)


def banner(note='',        # what the session has to say for itself: the model, or why there is none
           width=None,     # cells the art may use, gutter already taken off. None for no limit
           palette=None):  # a palette to paint in, or the active one
    """The bow, the arrow and the wordmark as one themed `Text`, the arrow aimed at the name.

    Narrower than `BANNER_MIN` and the bow goes rather than wrap, because a wrapped bow is not art;
    narrower than the wordmark and only the one line the CLI printed before this existed is left.
    """
    p = palette or GRUVBOX
    bold, quiet = f"bold {p['fg0']}", p['gray']
    if width is not None and width < len(WORDMARK[0]) + 2:
        out = Text('RAMABANA', style=bold)
        if note: out.append(f'  {note}', style=quiet)
        return out
    if width is not None and width < BANNER_MIN:
        out = Text('\n', style=bold).join(Text(w, style=bold) for w in WORDMARK)
        if note: out.append(f'\n{note}', style=quiet)
        return out
    left = max(map(len, BOW)) + BANNER_GAP
    top = len(BOW) // 2 - 1        # the wordmark straddles the arrow's row, the note sits under it
    out = Text()
    for i, row in enumerate(BOW):
        if i: out.append('\n')
        word = top <= i < top + len(WORDMARK)
        beside = WORDMARK[i - top] if word else (note if i == top + len(WORDMARK) else '')
        out.append(_glyphs(row.ljust(left) if beside else row, p))
        if beside: out.append(beside, style=bold if word else quiet)
    return out


#: The arrow, tail first, and the strip it crosses in the status bar. A rotating ring was the first
#: try and it split the arrow in two at the wrap -- a head at one end of the strip and a tail at the
#: other, which reads as debris beside the word `working` rather than as anything flying. The arrow
#: grows in from the left instead and leaves at the right, so there is always exactly one head.
ARROW = '╴━▸'
MARK_WIDTH = len(ARROW) + 2
#: The same strip with the bow unstrung: no head on it, because there is nothing to loose.
SLACK = '┈┈┈'.ljust(MARK_WIDTH)


def arrow_mark(state='ready',   # working | ready | anything else, which is idle
               frame=0,         # the repaint counter; only `working` reads it
               palette=None):   # a palette to paint in, or the active one
    """The mark beside the state in the status bar, `MARK_WIDTH` cells wide whatever the state.

    A busy `Ui` repaints this ten times a second, so a mark that changed width would jitter the whole
    bar; every state is the same strip, and only the glyphs in it and their colour move.
    """
    p = palette or GRUVBOX
    if state == 'working':
        head, shaft = p['yellow'], p['orange']
        at = frame % MARK_WIDTH
        cells = [' '] * MARK_WIDTH
        for i, glyph in enumerate(ARROW):
            c = at - (len(ARROW) - 1 - i)
            if 0 <= c < MARK_WIDTH: cells[c] = glyph
        row = ''.join(cells)
    elif state == 'ready': row, head, shaft = ARROW.ljust(MARK_WIDTH), p['green'], p['green']
    else: row, head, shaft = SLACK, p['gray'], p['gray']
    return _spans((ch, f'bold {head}' if ch == '▸' else (shaft if ch in '╴━' else p['gray']))
                  for ch in row)


#: The arrow in flight, tail first: the trail thins and cools behind the head. One entry per cell.
FLIGHT = (('·', 'bg2'), ('·', 'gray'), ('┄', 'gray'), ('━', 'orange'), ('━', 'yellow'), ('▸', 'yellow'))
FLIGHT_WIDTH = 16


def flight_line(frame=0, width=FLIGHT_WIDTH, palette=None):
    "One row of the arrow crossing `width` cells, and `width` cells wide at every frame of it."
    p = palette or GRUVBOX
    width = max(len(FLIGHT), width)
    cells = [(' ', None)] * width
    at = frame % (width + len(FLIGHT))
    for i, (glyph, key) in enumerate(FLIGHT):
        c = at - (len(FLIGHT) - 1 - i)
        if 0 <= c < width: cells[c] = (glyph, f"bold {p['yellow']}" if glyph == '▸' else p[key])
    return _spans(cells)


In [ ]:
#| hide
# The emblem is drawn in two disjoint alphabets, which is what lets `_glyphs` colour the arrow
# apart from the bow without being told where either one is.
test_eq(set(_ARROW_GLYPHS) & set(_BOW_GLYPHS), set())
test_eq(set(''.join(WORDMARK).strip()) & (set(_ARROW_GLYPHS) | set(_BOW_GLYPHS)), set())

# The full banner is the bow's rows, and no row of it overruns the width it asked for.
full = banner('gemma-e2b', 100)
test_eq(len(full.plain.split('\n')), len(BOW))
assert max(cell_len(r) for r in full.plain.split('\n')) <= BANNER_MIN + len('gemma-e2b')

# A terminal too narrow for the bow gets the wordmark, and one too narrow for that gets one line.
test_eq(len(banner('', BANNER_MIN - 1).plain.split('\n')), len(WORDMARK))
test_eq(banner('', 20).plain, 'RAMABANA')


In [ ]:
#| hide
# Every state is the same width, because the status bar repaints ten times a second and a mark
# that grew by a cell would shove the model name and the cost sideways on every frame.
_w = lambda t: cell_len(t.plain)
test_eq({_w(arrow_mark('working', f)) for f in range(MARK_WIDTH * 3)}, {MARK_WIDTH})
test_eq(_w(arrow_mark('ready')), MARK_WIDTH)
test_eq(_w(arrow_mark('idle')), MARK_WIDTH)

# One head per frame, always. A rotating ring was the first try and it split the arrow at the wrap.
test_eq({arrow_mark('working', f).plain.count('▸') for f in range(MARK_WIDTH * 3)}, {1})
test_eq(arrow_mark('working', 0).plain, arrow_mark('working', MARK_WIDTH).plain)

# The flight track is constant-width too, and the arrow really does cross it.
test_eq({_w(flight_line(f, 20)) for f in range(60)}, {20})
assert '▸' in flight_line(0, 20).plain and '▸' in flight_line(10, 20).plain
test_ne(flight_line(0, 20).plain, flight_line(3, 20).plain)


In [ ]:
from rich.console import Console
_c = Console(force_terminal=True, width=100)
for _t in ('dark', 'gruvbox', 'nord', 'tokyonight'):
    set_theme(_t)
    _c.print(banner(f"{_t} · rama's arrow, on the string", 100))
    _c.print(arrow_mark('ready'), arrow_mark('working', 1), flight_line(9, 24))
set_theme('dark')


In [ ]:
#| export
BUILD = datetime.fromtimestamp(Path(__file__).stat().st_mtime).strftime('%H:%M') if '__file__' in dir() else ''
VERSION = f'{__version__}+{BUILD}' if BUILD else __version__

In [ ]:
#| hide
import ramabana.cli as _c
from ramabana import __version__
assert _c.VERSION.startswith(__version__)
assert ':' in _c.BUILD
test_eq(VERSION, __version__)      # ...and in here it degrades to the plain version

In [ ]:
#| export
GUIDE = """START
  ramabana ["question"]   interactive session or one turn
  --python                    start a protected Python prompt
  --attach NAME  --kernels     join or list live dhrishti sessions
  --theme NAME                 a palette; /theme lists them and next steps
MODES
  /python  /agent  /vars  /promote NAME
  Python enter runs complete code; tab completes; ctrl+c interrupts.
  The agent reads your namespace but writes only to its own overlay.
READING A TURN
  Narration, calls and the answer land in the order they happened.
  ctrl+o folds or opens all the working; alt+1..9 opens one entry.
  ctrl+r browses, searches and copies blocks; /mouse to click them.
WORK
  /plan  /todo  /sessions  /resume [ID|latest]  /model [NAME]
  --max-tool-calls auto|N  --max-steps auto|N  /tool-budget  /steps
FILES AND API
  drop or paste media, write @path, or /attach PATH; /detach drops it.
  --spec enables api_load, api_ops, and api_call.

  /quit or /exit leaves; ctrl+c stops a turn and keeps the session."""

In [ ]:
#| export
def key_card(text):
    "The key list, with its labels and separators styled."
    out = Text()
    for line in text.splitlines():
        label, _, rest = line.partition('  ')
        out.append(f'{label:<11}', style=f"bold {GRUVBOX['yellow']}")
        for i, seg in enumerate(rest.split('·')):
            if i: out.append('· ', style=GRUVBOX['gray'])
            out.append(seg.strip() + ' ', style=GRUVBOX['fg1'])
        out.append('\n')
    return out

def guide_text(text):
    "The guide, with its section headers picked out and the commands in the key colour."
    out = Text()
    for line in text.splitlines():
        if line and not line[0].isspace():
            out.append(line + '\n', style=f"bold {GRUVBOX['yellow']}")
        elif line.startswith('  ') and (s := line.strip()) and (s[0] in '/-' or s.startswith('ramabana')):
            head, sep, rest = line.partition('   ')
            out.append(head, style=GRUVBOX['aqua'])
            out.append(sep + rest + '\n', style=GRUVBOX['fg1'])
        else: out.append(line + '\n', style=GRUVBOX['fg1'])
    return out

In [ ]:
print(HELP)

normal  enter send · tab complete /commands · ctrl+t plan · ctrl+p/n history · ↑/↓ or ctrl+r transcript · ctrl+o fold the working · alt+1..9 drill in · ctrl+c stop · ctrl+d quit
timeline  a turn reads top to bottom · ┆ narration · │ a call · the answer last · ctrl+o all the working · alt+1..9 one entry
transcript  ↑/↓ blocks · pgup/pgdn page · /? search · n/N matches · g/G ends · y copy block · i compose · esc leave
edit    ctrl+a/e ends · ctrl+u/k cut line · ctrl+w cut word · ctrl+y yank
media   drop or paste a path to attach · @path in a prompt · /attach PATH · /detach [N] · ctrl+v or /paste clipboard image
copy    select with the mouse as in any scrollback · /copy the last reply · /copy turn for all of it · ctrl+r then y for any block
approve y approve · n refuse · a approve all · ctrl+y approve with a note · or type a reason and press enter to refuse
options ↑/↓ move · enter choose · an option's own letter picks it · esc cancel and keep the line
python  /python takes the line · /ag

In [ ]:
print(guide_text(GUIDE))

START
  ramabana ["question"]   interactive session or one turn
  --python                    start a protected Python prompt
  --attach NAME  --kernels     join or list live dhrishti sessions
  --theme auto|dark|light      choose the terminal palette
MODES
  /python  /agent  /vars  /promote NAME
  Python enter runs complete code; tab completes; ctrl+c interrupts.
  The agent reads your namespace but writes only to its own overlay.
READING A TURN
  Narration, calls and the answer land in the order they happened.
  ctrl+o folds or opens all the working; alt+1..9 opens one entry.
  ctrl+r browses, searches and copies blocks; /mouse to click them.
WORK
  /plan  /todo  /sessions  /resume [ID|latest]  /model [NAME]
  --max-tool-calls auto|N  --max-steps auto|N  /tool-budget  /steps
FILES AND API
  drop or paste media, write @path, or /attach PATH; /detach drops it.
  --spec enables api_load, api_ops, and api_call.

  /quit or /exit leaves; ctrl+c stops a turn and keeps the session.



## Attachments

A prompt can carry pictures and sound, and it is asked to carry them the way a terminal already hands files over: drop one on the window, paste a path, or name it with `@path`. Every shape a terminal delivers a dropped file in resolves to the same attachment. Quoted, bracketed, a `file://` URI, spaces backslash-escaped. Because a person who drops a picture on the prompt has already said what they meant by it.

Pictures go to the model as content parts, and *every* attachment is additionally named by absolute path, which is the part that survives whatever the runtime can accept: a model with no ear for audio can still reach a `.wav` through the file tools.

In [ ]:
#| export
MEDIA = {
    '.png': ('image', 'image/png'),   '.jpg':  ('image', 'image/jpeg'),
    '.jpeg': ('image', 'image/jpeg'), '.gif':  ('image', 'image/gif'),
    '.webp': ('image', 'image/webp'),
    '.wav': ('audio', 'audio/wav'),   '.mp3':  ('audio', 'audio/mpeg'),
    '.m4a': ('audio', 'audio/mp4'),   '.ogg':  ('audio', 'audio/ogg'),
    '.flac': ('audio', 'audio/flac'), '.aac':  ('audio', 'audio/aac'),
}

MAX_MEDIA = 20 << 20
MAX_ATTACH = 8
CLIP_IMAGE = (('pngpaste', '-'),
              ('wl-paste', '--type', 'image/png'),
              ('xclip', '-selection', 'clipboard', '-t', 'image/png', '-o'))

def _human(n):
    "A byte count the way a person reads one."
    if n < 1024: return f'{n}B'
    for unit in ('KB', 'MB'):
        n /= 1024
        if n < 1024: return f'{n:.1f}{unit}'
    return f'{n / 1024:.1f}GB'

def media_path(s):
    "One path in whatever shape a terminal delivered it, or None."
    s = str(s).strip().strip('[]').strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in '"\'': s = s[1:-1]
    if s.startswith('file://'): s = unquote(urlparse(s).path)
    else: s = s.replace('\\ ', ' ')
    if not s: return None
    try: return Path(s).expanduser()
    except RuntimeError: return None

def is_media(p):
    "Whether `p` names a media file that exists."
    if p is None or p.suffix.lower() not in MEDIA: return False
    try: return p.is_file()
    except OSError: return False

def media_paths(text):
    "Every media file a paste names, or nothing when the paste is anything else."
    raw = str(text).strip().strip('[]').strip()
    if not raw: return []
    whole = media_path(raw)
    if is_media(whole): return [whole]        # one drop whose filename contains spaces
    try: toks = shlex.split(raw)
    except ValueError: return []
    paths = [media_path(t) for t in toks]
    return paths if paths and all(is_media(p) for p in paths) else []

#: `@path`, the file reference every other harness spells the same way.
ATTACH_REF = re.compile(r'(?<!\S)@(\S+)')

#: Punctuation a `@path` can pick up from the sentence it sits in, never from a filename.
TRAILING = '?!,;:.)]}\'"'

def attach_refs(text):
    """Media named `@path` inside a typed prompt.

    A reference at the end of a sentence carries the sentence's punctuation. Trailing marks
    come off one at a time until what is left names a file: `@shot.png?` is a question about a
    picture rather than a path to one.
    """
    out = []
    for m in ATTACH_REF.finditer(str(text or '')):
        tok = m.group(1)
        while tok:
            p = media_path(tok)
            if is_media(p):
                out.append(p)
                break
            if tok[-1] not in TRAILING: break
            tok = tok[:-1]
    return out

def clipboard_png():
    "A picture on the system clipboard as PNG bytes, or None when there is not one."
    for cmd in CLIP_IMAGE:
        if shutil.which(cmd[0]) is None: continue
        try: out = subprocess.run(cmd, capture_output=True, timeout=5).stdout
        except Exception: continue
        if out[:8] == b'\x89PNG\r\n\x1a\n': return out
    return None

class Attachment:
    """One media file riding along with the next prompt.
    The bytes are read once, when the file is attached: what gets sent is then what was named
    and measured on screen, even if the file changes or goes away before the turn.
    """
    def __init__(self, path):
        self.path = Path(path).expanduser().resolve()
        self.kind, self.mime = MEDIA[self.path.suffix.lower()]
        self.data = self.path.read_bytes()

    @property
    def name(self): return self.path.name
    def __len__(self): return len(self.data)
    def label(self): return f'{self.name} ({_human(len(self))})'
    def line(self): return f'{self.kind}  {self.path}  {self.mime}  {_human(len(self))}'
    def __repr__(self): return f'Attachment({self.kind} {self.label()})'

def sendable(atts, spec=None):
    "The attachment kinds `spec`'s model can be sent. Pictures always. Sound where it can hear."
    kinds = {'image'}
    if spec is None or accepts(spec, 'audio'): kinds.add('audio')
    return kinds

def media_parts(atts, spec=None):
    "The attachments that go out as model content parts."
    ok = sendable(atts, spec)
    return [a.data for a in atts if a.kind in ok]

def media_note(atts, spec=None):
    "What the message says about the files attached to it."
    if not atts: return ''
    ok = sendable(atts, spec)
    rows, note = '\n'.join(a.line() for a in atts), ''
    if any(a.kind == 'image' for a in atts): note += '\nThe images above are attached to this message.'
    if any(a.kind == 'audio' for a in atts):
        note += ('\nThe audio above is attached to this message.' if 'audio' in ok else
                 '\nAudio is attached by path only -- this model does not accept audio input. '
                 'Read it from the path above if you need its contents.')
    return f'\n\n<attachments>\n{rows}\n</attachments>{note}'

### Pictures a model sent back

Generated images arrive on `rishi`'s `Resp.media`, are saved beside the session, then drawn with `kittytgp` where the terminal speaks kitty graphics.

kittytgp places the image as `U+10EEEE` placeholder *text*, which is why it works here: teleprint's compositor redraws from its model. Text rows survive a repaint where a raw inline-image escape would not. Elsewhere, and for non-PNG, the path is printed.

In [ ]:
#| export
#: Terminals that speak kitty's graphics protocol, by what they put in the environment.
KITTY_ENV = ('KITTY_WINDOW_ID', 'GHOSTTY_RESOURCES_DIR')
KITTY_TERM = ('kitty', 'kaku', 'ghostty', 'wezterm')        # substrings of $TERM
KITTY_PROGRAM = ('WezTerm', 'Kaku', 'ghostty', 'kitty')     # exact $TERM_PROGRAM

def kitty_graphics():
    "Does this terminal speak the kitty graphics protocol?"
    if (forced := env('KITTY')) is not None:
        return str(forced).strip().lower() not in ('0', 'false', 'no', '')
    if any(os.environ.get(k) for k in KITTY_ENV): return True
    term, prog = os.environ.get('TERM', '').lower(), os.environ.get('TERM_PROGRAM', '')
    return any(t in term for t in KITTY_TERM) or prog in KITTY_PROGRAM

#: Widest a picture is drawn, how tall a cell is relative to its width, and how many one turn
#: may draw. Small and few on purpose: a tall block is what makes the transcript hard to scroll.
MAX_IMG_COLS = 24
CELL_ASPECT = 2.1
MAX_IMG_DRAW = 2

def png_size(path):
    "`(width, height)` in pixels from a PNG's IHDR, or `None`."
    try: b = Path(path).read_bytes()[:24]
    except OSError: return None
    if len(b) < 24 or b[:8] != b'\x89PNG\r\n\x1a\n': return None
    w, h = int.from_bytes(b[16:20], 'big'), int.from_bytes(b[20:24], 'big')
    return (w, h) if w and h else None

def img_cells(path, cols):
    "Cell width and height for `path` drawn at most `cols` wide, keeping its aspect."
    if not (wh := png_size(path)): return None
    w, h = wh
    c = max(1, min(cols, MAX_IMG_COLS))
    return c, max(1, round(c * (h / w) / CELL_ASPECT))

APC_CHUNK = 4096

def draw_png(path, cols=MAX_IMG_COLS):
    "The escape that draws `path` at the cursor, or `''`."
    if not kitty_graphics() or Path(path).suffix.lower() != '.png': return ''
    if not (cr := img_cells(path, cols)): return ''
    try: data = b64encode(Path(path).read_bytes())
    except OSError: return ''
    head, out = f'a=T,f=100,q=2,C=1,c={cr[0]},r={cr[1]}', []
    for i in range(0, len(data), APC_CHUNK):
        piece, more = data[i:i+APC_CHUNK], int(i + APC_CHUNK < len(data))
        ctl = f'{head},m={more}' if i == 0 else f'm={more}'
        out.append('\x1b_G' + ctl + ';' + piece.decode() + '\x1b\\')
    return ''.join(out)

def media_line(path):
    "What the transcript says about a saved picture when it cannot draw it."
    return f'![{Path(path).name}]({path})' if kitty_graphics() else f'saved  {path}'

In [ ]:
import os as _os
_seen = {k: _os.environ.pop(k, None) for k in (*KITTY_ENV, 'TERM', 'TERM_PROGRAM')}
try:
    # no kitty in the environment: nothing is drawn, and the path is what the transcript gets
    test_eq(kitty_graphics(), False)
    test_eq(draw_png('/tmp/nope.png'), '')
    assert media_line('/tmp/a.png').startswith('saved  ')

    import tempfile
    with tempfile.TemporaryDirectory() as d:
        png = b'\x89PNG\r\n\x1a\n' + b'x'
        p = save_media({'mime': 'image/png', 'data': png}, d)
        test_eq(p.name, 'image-1.png')
        test_eq(p.read_bytes(), png)
        test_eq(p.parent.name, 'media')
        # a second picture in the same turn does not overwrite the first
        test_eq(save_media({'mime': 'image/png', 'data': png}, d).name, 'image-2.png')
        # a mime with no entry in the table still lands somewhere sensible
        test_eq(save_media({'mime': 'image/avif', 'data': png}, d).suffix, '.avif')
finally:
    for k, v in _seen.items():
        if v is not None: _os.environ[k] = v

In [ ]:
#| export
MAX_FILE_ATTACH = 120_000  # characters. Enough source to be useful without consuming a whole turn
def file_refs(text):
    "Relative paths named as `@path` in a prompt, with sentence punctuation removed."
    out = []
    for m in ATTACH_REF.finditer(str(text or '')):
        tok = m.group(1)
        while tok and tok[-1] in TRAILING: tok = tok[:-1]
        if tok: out.append(tok)
    return out

class FileAttachment:
    "One text or notebook source attachment, captured when the prompt is submitted."
    kind = 'file'
    def __init__(self, path, text): self.path, self.data = Path(path), str(text)
    @property
    def name(self): return self.path.name
    def __len__(self): return len(self.data)
    def label(self): return f'{self.name} ({_human(len(self))})'
    def line(self): return f'file  {self.path}  text/plain  {_human(len(self))}'

def file_note(atts):
    "The grounded text from selected workspace files, with their exact paths."
    files = [a for a in atts if a.kind == 'file']
    if not files: return ''
    return '\n\n<attached-files>\n' + '\n\n'.join(f'<file path="{a.path}">\n{a.data}\n</file>' for a in files) + '\n</attached-files>'

In [ ]:
media_dir = Path(tempfile.mkdtemp())
(media_dir / 'shot.png').write_bytes(b'\x89PNG\r\n\x1a\n' + b'0' * 64)
(media_dir / 'My Photo.png').write_bytes(b'\x89PNG\r\n\x1a\n')
(media_dir / 'note.wav').write_bytes(b'RIFF' + b'0' * 40)
shot, spaced, snd_path = media_dir / 'shot.png', media_dir / 'My Photo.png', media_dir / 'note.wav'
sorted(p.name for p in media_dir.iterdir())

['My Photo.png', 'note.wav', 'shot.png']

### UI attachment methods are defined after `Ui`, below.

In [ ]:
test_eq(media_paths(str(shot)), [shot])                            # a bare path
test_eq(media_paths(f'"{shot}"'), [shot])                          # quoted
test_eq(media_paths(f"'{shot}'"), [shot])
test_eq(media_paths(f'[{shot}]'), [shot])                          # bracketed by the terminal
test_eq(media_paths(f'file://{shot}'), [shot])                     # a URI, as GNOME and KDE send
test_eq(media_paths(f'  {shot}  '), [shot])
test_eq(media_paths(str(spaced)), [spaced])                        # a name with a space in it
test_eq(media_paths(str(spaced).replace(' ', '\\ ')), [spaced])    # ...as macOS escapes it
test_eq(media_paths(f'{shot} {snd_path}'), [shot, snd_path])       # two files dropped together

In [ ]:
test_eq(media_paths('where is the threshold?'), [])
test_eq(media_paths(f'have a look at {shot} when you get a chance'), [])   # prose stays prose
test_eq(media_paths(str(media_dir / 'gone.png')), [])                     # a path to nothing
from unittest.mock import patch as mock_patch

_real_expanduser = Path.expanduser
def _expand_without_home(path):
    if str(path).startswith('~'): raise RuntimeError('Could not determine home directory.')
    return _real_expanduser(path)

with mock_patch.object(Path, 'expanduser', _expand_without_home):
    test_eq(media_paths('keep ~literal text'), [])
test_eq(attach_refs(f'compare @{shot} against the spec'), [shot])         # `@path`, as other harnesses spell it
test_eq(attach_refs(f'what is in @{shot}?'), [shot])   # not the sentence's question mark
test_eq(attach_refs('email me @ the address on file'), [])

In [ ]:
img, snd = Attachment(shot), Attachment(snd_path)
test_eq((img.kind, img.mime), ('image', 'image/png'))
test_eq((snd.kind, snd.mime), ('audio', 'audio/wav'))
test_eq(media_note([]), '')

# gemini hears, claude does not, and rishi's `model_caps` is what says so
hears, deaf = resolve('gemini/gemini-2.5-flash'), resolve('anthropic/claude-opus-4-5')
test_eq(media_parts([img, snd], hears), [img.data, snd.data])
test_eq(media_parts([img, snd], deaf), [img.data])      # sound goes by path alone
assert 'The audio above is attached' in media_note([img, snd], hears)
assert 'does not accept audio input' in media_note([img, snd], deaf)

# with no spec at all everything goes: not knowing must not quietly shrink the message
test_eq(media_parts([img, snd]), [img.data, snd.data])
note = media_note([img, snd])
assert str(img.path) in note and str(snd.path) in note
print(note)



<attachments>
image  /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp20esuni1/shot.png  image/png  72B
audio  /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp20esuni1/note.wav  audio/wav  44B
</attachments>
The images above are attached to this message.
The audio above is attached to this message.


## The turn

`agent.stream` is a blocking generator on the model's thread. Its chunks come back over a queue rather than being awaited. The loop has to stay free the entire time: an approval that cannot be answered until the turn ends is not an approval, and a tool call that cannot repaint until then is not a live feed.

In [ ]:
#| export
@dataclass
class Option:
    "One choice on the options row: the key that picks it, what it says, and what it asks for."
    key: str                # the letter that picks it directly
    label: str              # one word, in the column
    note: str = ''          # what choosing it means, in a few more
    suffix: str = ''        # appended to the prompt. `None` drops the prompt instead

REFACTOR = (
    Option('a', 'apply', 'edit the files, then run the project’s checks',
           '\n\nMake the change now. Read each file before you edit it, keep every edit narrow, then '
           'run the project’s own tests or linter with run_shell and report exactly what passed.'),
    Option('p', 'plan', 'read only, propose the steps',
           '\n\nDo not edit anything. Read the relevant code and reply with a numbered plan: the files '
           'and symbols involved, the change to each, its risk, and the order to do them in.'),
    Option('s', 'scope', 'name what is in and out, then stop',
           '\n\nAnswer only this, with no edits and no plan: which files and symbols are in scope for '
           'this change, and which nearby ones are deliberately not. One line each.'),
    Option('c', 'cancel', 'put the line back to edit', None),
)

MENUS = ((('refactor', 'restructure', 'reorganise', 'reorganize', 'rewrite this',
           'clean up this', 'tidy up this'), 'how should I take this?', REFACTOR),)


def options_for(text):
    "`(title, options)` for a prompt worth asking *how* about, or None for one that is not."
    p = str(text or '').lower()
    return next(((title, opts) for words, title, opts in MENUS if any(w in p for w in words)), None)


class ChoiceMenu:
    "The options row: one line per `Option`, above the input line."

    def __init__(self, title, options): self.title, self.options, self.at = title, list(options), 0

    def render(self):
        rows = [Text(f' {self.title}', style=f"bold {GRUVBOX['fg0']}")]
        width = max(len(o.label) for o in self.options)
        for i, o in enumerate(self.options):
            on = i == self.at
            row = Text(' › ' if on else '   ', style=f"bold {GRUVBOX['yellow']}")
            row.append(f'{o.key}  ', style=f"bold {GRUVBOX['yellow'] if on else GRUVBOX['gray']}")
            row.append(o.label.ljust(width), style=GRUVBOX['fg0'] if on else GRUVBOX['fg1'])
            if o.note: row.append(f'   {o.note}', style=GRUVBOX['gray'])
            rows.append(row)
        return rows

    def choose(self, key):
        "`(done, option)` for one keystroke. A `None` option means the prompt was dropped."
        if key in ('escape', 'ctrl+c'): return True, None
        if key in ('up', 'left'): self.at = (self.at - 1) % len(self.options)
        elif key in ('down', 'right'): self.at = (self.at + 1) % len(self.options)
        elif key == 'enter': return True, self.options[self.at]
        elif key.isdigit() and 0 < int(key) <= len(self.options): return True, self.options[int(key) - 1]
        elif (hit := next((o for o in self.options if o.key == key.lower()), None)) is not None: return True, hit
        return False, None


async def run_turn(ui, prompt):
    """One turn, streamed into the transcript.

    The agent's `stream` is a blocking generator on the model's own thread. The chunks
    come back over a queue rather than being awaited: the loop has to stay free the whole
    time, or an approval could never be answered and a tool call could never repaint.

    The attachments are taken here, at the start, rather than released at the end: the prompt
    that named them is then the only one that carries them, however the turn goes.
    """
    loop, q = asyncio.get_running_loop(), asyncio.Queue()
    ui.log_cell('**user**\n\n' + prompt, cell_type='markdown')
    ui._reply, ui._seg, ui._seg_blk, ui._rendered = '', '', None, ''
    ui._plan_blk = None        # a new turn prints the plan once more, then rewrites that one
    ui._turn_at, ui._turn_from = time.monotonic(), next(reversed(ui.comp.blocks), 0)
    atts, ui.attachments = list(ui.attachments), []
    spec = ui.agent.model
    ask, media = prompt + media_note(atts, spec), media_parts(atts, spec)
    def pump():
        try:
            for chunk in ui.agent.stream_with(ask, image=media or None):
                loop.call_soon_threadsafe(q.put_nowait, chunk)
        except Exception as e: loop.call_soon_threadsafe(q.put_nowait, agent_err(e))
        finally: loop.call_soon_threadsafe(q.put_nowait, None)
    threading.Thread(target=pump, daemon=True).start()
    blk, run = None, None
    try:
        while True:
            try: chunk = await asyncio.wait_for(q.get(), .05)
            except asyncio.TimeoutError:
                run = run or ui.agent.run()
                if run is not None and run.terminal: break
                continue
            if chunk is None:break
            run = run or ui.agent.run()
            if run is not None and run.cancelled:break
            blk = ui.stream(blk, chunk)
    finally:
        ui.turn = None
        ui.flush_stream()          # the throttle may still owe the last chunk a render
        ui._seg_blk = None         # the answer is final: no later chunk grows it
        ui.show_media(ui.agent.last_media)
        for p in ui.agent.problems: ui.say(Text(p), 'error')
        ui.agent.clear_problems()
        ui.touch(now=True)
        ui.paint()
    if blk is not None and ui._reply: ui.log_cell('**assistant**\n\n' + ui._reply, cell_type='markdown')
    return blk

## The surface

`Ui` is the whole terminal surface, and every method on it is synchronous and free of tty work. This is what lets the tests below drive it against an emulated terminal instead of mocking one. The one hazard it has to handle is thread affinity: activity and approval callbacks arrive on the model's worker thread, and a compositor may only be touched from the loop thread. Everything they do goes through `_post`.

It carries three responsibilities: painting the tail (status bar plus input line), turning tool calls and approval requests into blocks, and deciding what one keystroke means. This is either a coroutine for the caller to spawn, `'quit'`, or nothing.

In [ ]:
#| export
class Ui:
    """The terminal surface: a transcript of blocks, a status bar, and one line to type in.

    Every method here is synchronous and free of tty work. The whole surface can be
    driven in a test against an emulated terminal. Which is why the async loop below is
    as small as it is.

    Callbacks arrive from the model's worker thread (`Activity.on_change`, `Approvals`), and
    a compositor may only be touched from the loop thread. Everything they do goes
    through `_post`. Without a loop registered it calls straight through, which is what
    makes the synchronous tests possible.
    """

    def __init__(self, comp, agent, loop=None):
        self.comp, self.agent, self.loop = comp, agent, loop
        comp.console.push_theme(MARKDOWN_THEME)
        self.buf = Buffer()
        self.ask = None            # the `Ask` waiting on an answer, or None
        self.turn = None           # the running turn's task, or None
        self.acts = {}             # act id -> its block, for calls that have one of their own
        self.by_id = {}            # act id -> the `Act`, while it may still need redrawing
        self.kids = {}             # delegate act id -> the calls its sub-agent has made
        self.hint = ''
        self.mode = 'agent'        # 'python' once `enter_python` has a kernel
        self.kernel = None         # the owner's `pyrepl.Kernel`, or None
        self.attached = ''
        self._join_ask = ''        # the session /join asked about. The same one again means yes
        self._suggesting = False   # one completion in flight at a time
        self.desc = []             # 'name -> type' for the live names among the candidates
        self.attachments = []      # `Attachment`s the next prompt carries
        self.frame = 0             # animated status frame. Advanced only while a turn runs
        self._reply = ''           # every word this turn has said, for the notebook log and `/copy`
        self._seg = ''             # the current prose segment: the text of one step of the timeline
        self._seg_blk = None       # the block that segment grows in, or None between segments
        self._painted_at = 0.0     # when that segment last re-rendered, for `STREAM_EVERY`
        self._acted_at = 0.0       # when the tool pane last repainted, for `ACT_EVERY`
        self._held = set()         # blocks the reader folded or opened by hand
        self._turn_at = 0.0        # when the running turn began, for the working footer's clock
        self._turn_from = 0        # the block id the running turn started at, for `turn_blocks`
        self._rendered = ''        # the segment text last actually rendered, so a flush is never wasted
        self._plan_blk = None      # the turn's one plan block, updated in place. `on_plan` fires
        self._touched = 0.0        # when the transcript view last rebuilt, for `touch`
        self.history, self.history_at, self.draft = [], 0, ''
        self.menu = None            # the open `ChoiceMenu`, or None
        self.menu_prompt = ''       # the line it is asking about
        self.complete = None        # slash-command `CompletionMenu`, or None
        self.show_plan = bool(agent.plan)  # plan tooltip above the tail
        self.mouse = False         # whether the main screen takes the mouse. `/mouse` toggles it
        self._stop_at, self._stop_count, self._stop_run = 0., 0, ''
        self.transcript = TranscriptView(comp, self.tail)
        agent.activity.on_change = self.on_act
        agent.on_plan = self.on_plan
        if agent.approvals is not None: agent.approvals.listen(self.on_ask, self.on_answer)

    def _post(self, fn, *a):
        "Run `fn` on the loop thread, or now when there is no loop (a test, or startup)."
        if self.loop is None: return fn(*a)
        self.loop.call_soon_threadsafe(fn, *a)


    def status(self):
        "The status bar: what is loaded, whether it is working, and what it has cost."
        s = self.agent.status()
        busy = self.turn is not None or s['busy']
        state = 'working' if busy else ('ready' if s['ready'] else 'idle')
        color = GRUVBOX['yellow'] if busy else GRUVBOX['green'] if s['ready'] else GRUVBOX['gray']
        out = Text('RAMABANA', style=f"bold {GRUVBOX['fg0']}")
        out.append(f" {VERSION}", style=GRUVBOX['gray'])
        out.append(f"  {s['model']}  ", style=GRUVBOX['gray'])
        out.append(arrow_mark(state, self.frame))
        out.append(' ' + state.lower(), style=color)   # the ring may end on a glyph, so space it
        bits = [f"{s['ntools']} tools", f"{s['nskills']} skills", f"{round(s['pct_full'] * 100)}% ctx"]
        if s.get('plan_line'): bits.append(s['plan_line'])
        if s['compactions']: bits.append(f"{s['compactions']} compacted")
        if self.agent.use.total: bits.append(s['usage'])
        if s['problems']: bits.append(f"{len(s['problems'])} problems")
        out.append('  ' + ' · '.join(bits), style=GRUVBOX['gray'])
        if self.kernel is not None:
            out.append(f'  · {self.mode}', style=GRUVBOX['aqua'] if self.mode == 'python' else GRUVBOX['blue'])
        return out

    def turn_blocks(self):
        """The blocks of the turn on screen: everything printed since its prompt went up.

        Teleprint commits a block only when a borrow ends its epoch, which nothing in an ordinary
        session does, so `not committed` is every block since startup rather than every block of
        this turn. Folding or drilling by that reaches back through the whole session -- and
        unfolding twenty turns of tool results at once pushes thousands of rows across the top
        edge, where they ink into scrollback and no keystroke can take them back.
        """
        return [b for b in self.comp.blocks.values() if b.id >= self._turn_from and not b.committed]

    def drillable(self):
        """The foldable entries of the turn on screen, newest first: what alt+1..9 reaches.

        Teleprint has its own alt-digit numbering, but it stamps the digit into the gutter and needs
        one at least three glyphs wide; these gutters are two, and widening every one of them to
        carry a digit is a bigger change to how the surface looks than a drill-in is worth. The
        numbers live in the footer instead, where the eye already is while a turn runs.
        """
        return [b for b in reversed(self.turn_blocks())
                if b.tag in ('step', 'tool') and b.height > 1][:9]

    def drill(self, n):
        "Toggle the `n`th newest foldable entry, counting from 1. False when there is no such entry."
        self.flush_stream()
        blocks = self.drillable()
        if not 1 <= n <= len(blocks): return False
        blk = blocks[n - 1]
        self.comp.toggle(blk)
        self._held.add(blk.id)      # the reader has decided about this one; stop re-deciding
        self.touch(now=True)
        return True

    def working(self):
        "Where the model is at: the last few calls, then one line of totals. Only while a turn runs."
        # In the tail, so it never scrolls; tail rows never ink, so none of it reaches the transcript.
        if self.turn is None and not self.agent.busy: return []
        acts = self.agent.activity.since()
        nums = {b.id: i + 1 for i, b in enumerate(self.drillable())}
        rows = []
        for a in acts[-ACT_TAIL:]:
            style = GRUVBOX['yellow'] if not a.done else GRUVBOX['gray'] if a.ok else GRUVBOX['red']
            parent = self.acts.get(a.parent_action_id) if a.parent_action_id else None
            blk = parent if parent is not None else self.acts.get(a.id)
            n = nums.get(blk.id) if blk is not None else None
            t = Text(f'{n} ' if n else '  ', style=GRUVBOX['blue'] if n else GRUVBOX['gray'])
            t.append(('  ' if parent is not None else '') + a.line(), style=style)
            rows.append(t)
        mine = [a for a in acts if not a.parent_action_id or a.parent_action_id not in self.acts]
        bits = [f'step {len(mine)}' if mine else 'thinking']
        if self._turn_at: bits.append(f'{time.monotonic() - self._turn_at:.0f}s')
        if (sub := len(acts) - len(mine)): bits.append(f'{sub} delegated')
        if (n := sum(1 for a in acts if not a.done)) > 1: bits.append(f'{n} in flight')
        # The arrow itself, flying, on the row the eye is already reading the numbers off -- and
        # *after* them, because this row starts with `  step n` and four tests in `test_timeline`
        # say so. It gives way entirely on a terminal too narrow to hold both.
        line = Text('  ' + ' · '.join(bits), style=GRUVBOX['gray'])
        track = min(FLIGHT_WIDTH, self.comp.console.width - cell_len(line.plain) - 6)
        if track >= len(FLIGHT):
            line.append('   ')
            line.append(flight_line(self.frame, track))
        rows.append(line)
        return rows

    async def animate(self):
        "Repaint the live tail while a turn is running. The transcript remains untouched."
        while True:
            await asyncio.sleep(0.1)
            if self.turn is not None:
                self.frame += 1
                self.flush_stream()   # a model that stalls mid-prose must not leave its last words unseen
                self.paint()

    ASKING, PY_LABEL, CONT = 'approve? [y/n/a, or a reason + enter] ', 'python › ', '...      '

    def prompt(self):
        "The input line: an approval question when one is pending, otherwise the prompt."
        if self.ask is not None:
            return Text(self.ASKING, style=f"bold {GRUVBOX['yellow']}") + Text(self.buf.text, style=GRUVBOX['fg0'])
        if self.mode == 'python':
            from ramabana.pyrepl import hl
            body = Text('\n' + self.CONT, style=GRUVBOX['fg0']).join(hl(self.buf.text).split('\n', allow_blank=True))
            return Text(self.PY_LABEL, style=f"bold {GRUVBOX['aqua']}") + body
        return Text('▌ ', style=f"bold {GRUVBOX['blue']}") + Text(self.buf.text, style=GRUVBOX['fg0'])

    def _prefixed(self, text):
        "`text` with whatever precedes it on screen, which is what `tail` measures against."
        if self.ask is not None: return self.ASKING + text
        if self.mode == 'python': return self.PY_LABEL + text.replace('\n', '\n' + self.CONT)
        return '▌ ' + text

    def _suggest_soon(self):
        "Ask the kernel what the token could be, as a slash line lists while you type."
        if self.mode != 'python' or self.kernel is None or self._suggesting: return
        if self.turn is not None or self.ask is not None: return
        tok = re.split(r'[^\w.]', self.buf.text[:self.buf.cursor])[-1]
        if len(tok) < 2 or tok.endswith('..'): return
        self._suggesting = True
        self.comp.spawn(self._suggest(), name='suggest')

    def overlay(self):
        "Transient rows above the tail: options, slash completion, or the plan checklist."
        if self.menu is not None: return self.menu.render()
        if self.complete is not None:
            rows = [self.complete.renderable()]
            if self.desc: rows.append(Text('  '.join(self.desc), style=GRUVBOX['gray']))
            return rows
        if self.show_plan and self.agent.plan:
            return [Tooltip(self.agent.plan.md(), max_lines=12).renderable()]
        return []

    def paint(self):
        "Repaint the live tail, and the browsing view when it is the surface on screen."
        rows, cursor = self.tail()
        self.comp.set_tail(*rows, cursor=cursor, over=self.overlay())
        if self.transcript.active: self.transcript.draw()

    def touch(self, now=False):
        "New or changed blocks: a following transcript view tracks them, as `less +F` would."
        if now: self.flush_stream()
        if not self.transcript.active: return
        t = time.monotonic()
        if not now and t - self._touched < NOTIFY_EVERY: return
        self._touched = t
        self.transcript.notify()


    def say(self, body, kind='reply', fold=FOLD, source=None, pad=False):
        "Print one block. Strings stay literal. Explicit Rich renderables keep their styling."
        if source is None:
            if isinstance(body, str): source = body
            elif isinstance(body, Text): source = body.plain
        body = Text(body) if isinstance(body, str) else body
        blk = self.comp.print_block(body, gutter=GUTTERS.get(kind, GUTTERS['reply']),
                                    tag=kind, collapse_at=fold, source=source, pad=pad)
        if kind != 'reply' and (reply := self._seg_blk) is not None:
            self.comp._epoch.remove(reply.id)
            self.comp._epoch.append(reply.id)
            self.comp.blocks.pop(reply.id)
            self.comp.blocks[reply.id] = reply
            self.comp._frame()
        self.touch(now=True)
        return blk

    def note(self, text, kind='note'):
        "Print one unfolded `kind` block and stay on this side of the model. Always None."
        self.say(Text(text), kind, fold=None)
        return None

    def _close_seg(self):
        "End the open prose segment, so what comes next prints below it rather than above."
        blk = self._seg_blk
        if blk is None: return
        self.flush_stream()
        self._seg_blk, self._seg = None, ''
        blk.tag, blk.gutter, blk.collapse_at = 'step', GUTTERS['step'], FOLD_STEP
        if blk.height > FOLD_STEP: blk.collapsed = True
        self.comp.refresh_block(blk)

    def fold_work(self):
        "Open every step and call of the turn on screen, or shut them all again; returns which it did."
        self.flush_stream()
        work = [b for b in self.turn_blocks() if b.tag in ('step', 'tool') and b.height > 1]
        if not work: return False
        shut = not all(b.collapsed for b in work)
        for b in work:
            b.collapsed = shut
            self._held.add(b.id)    # ctrl+o is a decision too
            self.comp._dirty(b)
        self.comp._frame()
        self.touch(now=True)
        return shut

    def on_act(self, act):
        "Called twice per tool call, from the model's thread: once running, once finished."
        self._post(self._act, act)

    def _act_style(self, act):
        "Blue while it runs, green when it worked, red when it did not."
        return GRUVBOX['blue'] if not act.done else GRUVBOX['green'] if act.ok else GRUVBOX['red']

    def _folded(self, act, blk):
        "Whether a call folds to its summary row."
        if blk.id in self._held: return blk.collapsed
        if blk.height <= FOLD_TOOL: return False
        if not act.done: return FOLD_RUNNING and act.id not in self.kids
        return act.ok                          

    def start_turn(self, coro):
        "Spawn a turn unless one is already running, and say whether it started."
        if self.turn is not None:
            coro.close()
            self.note('a turn is running · ctrl+c stops it')
            return False
        self.turn = self.comp.spawn(coro, name='turn')
        return True

    def _act(self, act):
        if act.parent_action_id and act.parent_action_id in self.acts: return self._nest(act)
        self.by_id[act.id] = act
        blk = self.acts.get(act.id)
        if blk is None:
            self._close_seg()
            line = Text(act.line(), style=self._act_style(act))
            self.acts[act.id] = self.say(line, 'tool', source=act.line())
            return self.paint()
        if act.id in self.kids: self._paint_group(act.id)
        else:
            line = Text(act.line(), style=self._act_style(act))
            src = act.line() if not act.detail else f'{act.line()}\n{act.detail}'
            body = [line] if not act.detail else [line, Text(act.detail, style=GRUVBOX['gray'])]
            self.comp.set_body(blk, *body, source=src)
            blk.collapsed = self._folded(act, blk)
            self.comp.refresh_block(blk)
            if act.done: self.by_id.pop(act.id, None)
        self.touch()
        now = time.monotonic()
        if act.done or now - self._acted_at >= ACT_EVERY:
            self._acted_at = now
            self.paint()

    def _nest(self, act):
        "A sub-agent's call, folded into the delegate that asked for it rather than printed beside it."
        kids = self.kids.setdefault(act.parent_action_id, [])
        if not any(k.id == act.id for k in kids): kids.append(act)
        self._paint_group(act.parent_action_id)
        self.touch()
        return self.paint()

    def _paint_group(self, pid):
        "Redraw a delegate and everything its sub-agent has done so far, as one foldable block."
        if pid not in self.by_id: return   # its parent has been pruned; nothing left to redraw into
        act, blk, kids = self.by_id[pid], self.acts[pid], self.kids.get(pid, [])
        head = Text(act.line(), style=self._act_style(act))
        if kids: head.append(f'  · {len(kids)} call{"" if len(kids) == 1 else "s"}', style=GRUVBOX['gray'])
        shown, body = kids[-MAX_GROUP_ROWS:], [head]
        if len(kids) > len(shown):
            body.append(Text(f'   … {len(kids) - len(shown)} earlier', style=GRUVBOX['gray']))
        body += [Text('   ' + k.line(), style=self._act_style(k)) for k in shown]
        if act.detail: body.append(Text(act.detail, style=GRUVBOX['gray']))
        src = '\n'.join([act.line()] + ['   ' + k.line() for k in kids] + ([act.detail] if act.detail else []))
        self.comp.set_body(blk, *body, source=src)
        blk.collapsed = self._folded(act, blk)   # unfolded while it runs: you watch the sub-agent work
        self.comp.refresh_block(blk)


    def on_ask(self, ask):
        "A write is waiting on a person. Print what it would do, and take over the input line."
        self._post(self._ask, ask)

    def _ask(self, ask):
        self.ask = ask
        self.buf.clear()
        title = Text(ask.summary, style=f"bold {GRUVBOX['yellow']}")
        self.say(title + Text('\n\n') + Text(ask.preview, style=GRUVBOX['fg1']), 'ask', fold=None)
        self.paint()

    def on_answer(self, ask):
        self._post(self._answered, ask)

    def _answered(self, ask):
        if self.ask is not None and self.ask.id == ask.id: self.ask = None
        self.say(Text(answer_md(ask).replace('**', '')), 'note')
        self.paint()

    def answer(self, ok, session=False):
        """Answer the pending request, using whatever has been typed as the reason.

        A refusal with a reason is the point of the gate: it reaches the model, which can
        change approach instead of retrying the same edit.
        """
        if self.ask is None: return None
        note, self.buf.text = self.buf.text.strip(), ''
        return self.agent.approvals.answer(self.ask.id, ok, note, session=session)


    def attach(self, path):
        "Attach one media file to the next prompt, or say why it cannot be attached."
        p = media_path(path)
        if p is None: return 'nothing to attach'
        if p.is_dir(): return f'cannot attach {p.name}: it is a folder'
        if not p.exists(): return f'cannot attach {p.name}: no such file'
        if p.suffix.lower() not in MEDIA: return f'cannot attach {p.name}: {p.suffix or "no suffix"} is not media'
        if p.stat().st_size > MAX_MEDIA:
            return f'cannot attach {p.name}: {_human(p.stat().st_size)} is over the {_human(MAX_MEDIA)} limit'
        if len(self.attachments) >= MAX_ATTACH: return f'cannot attach {p.name}: {MAX_ATTACH} is the limit'
        a = Attachment(p)
        if any(x.path == a.path for x in self.attachments): return f'{a.name} is already attached'
        self.attachments.append(a)
        return f'attached {a.kind} {a.label()}'

    def detach(self, which=''):
        "Drop one attachment by 1-based index or name, or all of them when nothing is named."
        if not self.attachments: return 'nothing is attached'
        if not which:
            n, self.attachments = len(self.attachments), []
            return f'dropped {n} attachment' + ('s' if n > 1 else '')
        if which.isdigit() and 1 <= int(which) <= len(self.attachments):
            return f'dropped {self.attachments.pop(int(which) - 1).name}'
        hit = next((a for a in self.attachments if a.name == which), None)
        if hit is None: return f'not attached: {which}'
        self.attachments.remove(hit)
        return f'dropped {hit.name}'

    def attach_clipboard(self):
        "Attach a picture sitting on the system clipboard, by way of a temporary file."
        png = clipboard_png()
        if png is None:
            if not any(shutil.which(c[0]) for c in CLIP_IMAGE):
                return ('no clipboard image helper found: install pngpaste (macOS), '
                        'wl-clipboard or xclip (linux)')
            return 'no image on the clipboard'
        fd, name = tempfile.mkstemp(prefix='ramabana-paste-', suffix='.png')
        with os.fdopen(fd, 'wb') as f: f.write(png)
        return self.attach(name)

    def attach_row(self):
        "The chips above the prompt: what the next message will carry, or None when nothing will."
        if not self.attachments: return None
        row = Text(' ')
        for i, a in enumerate(self.attachments):
            if i: row.append('  ')
            row.append(f"{'◧' if a.kind == 'image' else '♪'}{i + 1} {a.label()}", style=GRUVBOX['aqua'])
        row.append('   /detach to drop', style=GRUVBOX['gray'])
        return row

    def copy_last(self, tag='reply'):
        """Put the newest `tag` block on the system clipboard with OSC 52, from the prompt.

        The transcript view's `y` copies the block under its cursor. This is the same reach
        for the case that is nearly always wanted. The answer that just arrived. Without
        leaving the prompt to get it.

        `/copy turn` is the other thing worth wanting: every word the last turn said, narration
        included, which since a turn became a timeline is no longer any one block.
        """
        tag = (tag or 'reply').strip() or 'reply'
        if tag == 'turn':   # every word of the turn, which a timeline has spread over several blocks
            if not self._reply: return 'no turn to copy'
            text = self._reply
        else:
            blk = next((b for b in reversed(self.turn_blocks()) if b.tag == tag), None)
            if blk is None: return f'no {tag} block in this turn to copy'
            text = self.transcript.block_text(blk)
        self.comp.tty.write('\x1b]52;c;' + b64encode(text.encode()).decode() + '\x07')
        return f'copied {len(text)} chars of the last {tag}'


    def on_plan(self, plan):
        "Repaint when the model (or a slash command) mutates the checklist."
        self.show_plan = bool(plan)
        self._post(self._paint_plan, plan)

    def _paint_plan(self, plan):
        "Keep one plan block for the turn and rewrite it"
        if not plan: return
        md = plan.md()
        if (blk := self._plan_blk) is None:
            self._plan_blk = self.say(plan_text(md), 'plan', fold=None, source=md)
            return
        self.comp.set_body(blk, plan_text(md), source=md)
        self.comp.refresh_block(blk)
        self.touch(now=True)
        self.paint()

    def _slash_matches(self):
        text = self.buf.text
        if not text.startswith('/') or ' ' in text: return None
        prefix = text[1:].lower()
        names = {*self.agent.commands(), *SURFACE_COMMANDS}
        hits = sorted(f'/{n}' for n in names if n.startswith(prefix))
        return hits or None

    def _refresh_complete(self):
        hits = self._slash_matches()
        if not hits:
            self.complete = None
            return
        self.complete = CompletionMenu(self.buf, hits, start=0, show=8)

    def guide(self):
        "The walkthrough behind `/guide`. A subclass adds its own surface to it."
        return GUIDE

    def submit(self):
        """Handle the typed line. Returns a coroutine for a turn, `'quit'`, or None when it was handled here.

        Most slash commands are answered by the agent. Every command the IDE has works
        here too. There is one implementation of `/model`, and it is not in a frontend. The
        ones kept here are the ones about this surface: its keys, its clipboard, its
        attachments, its mode. All of them are recognised *before* the options row, or a
        `/model` with the word "refactor" in it would open a menu instead of running.
        """
        src, line = self.buf.text, self.buf.text.strip()
        self.complete, self.desc = None, []
        self.buf.clear()
        if not line: return None
        if line in ('/python', '/py'): return self.enter_python()
        if line in ('/agent', '/a'):
            self.mode = 'agent'
            return self.note('agent mode' if self.kernel is not None else 'agent mode; no kernel is running')
        if line in ('/vars', '/v'):
            try: return self.note(self.agent.host.list_vars() or '(nothing bound yet)')
            except Exception as e: return self.note(agent_err(e), 'error')   # not every host has a session
        if (parts := line.split())[0] in ('/promote', '/adopt'):
            if len(parts) != 2: return self.note('usage: /promote NAME')
            return self._promote(parts[1])
        if self.mode == 'python' and not line.startswith('/'):
            from ramabana.pyrepl import hl
            # the raw buffer, not the stripped line. An indented paste keeps its first row
            self.say(hl(src), 'user', source=src)
            return self.run_code(src)
        if not line.startswith('/') and (opts := options_for(line)) is not None:
            self.menu_prompt, self.menu = line, ChoiceMenu(*opts)
            return None
        self.say(Text(line), 'user', pad=True)   # one blank row: a session reads as turns
        if line in ('/quit', '/exit', '/q'): return 'quit'
        if line == '/kernels':
            from ramabana.pyrepl import sessions
            return self.note(sessions())
        if (bits := line.split())[0] == '/join':
            if len(bits) != 2:
                from ramabana.pyrepl import sessions
                self.note(sessions())
                return self.note('usage: /join NAME-or-URL')
            return self._join(bits[1])
        if line in ('/help', '/?'): self.say(key_card(HELP), 'note', fold=None, source=HELP); return None
        if line == '/guide':
            self.say(guide_text(g := self.guide()), 'note', fold=None, source=g)
            return None
        name, _, arg = line.partition(' ')
        arg = arg.strip()
        if name in ('/attach', '/add'):
            if not arg: return self.note('usage: /attach PATH [PATH ...]')
            found = media_paths(arg) or [media_path(arg)]
            return self.note('\n'.join(self.attach(p) for p in found))
        if name == '/detach': return self.note(self.detach(arg))
        if name == '/paste': return self.note(self.attach_clipboard())
        if name == '/copy': return self.note(self.copy_last(arg))
        if name == '/mouse': return self.note(self.set_mouse(arg))
        if line.startswith('/'):
            out = self.agent.command(line)
            kind = 'plan' if name in ('/plan', '/todo', '/todos') and out is not None else (
                'note' if out is not None else 'error')
            self.say(Text(out) if out is not None else Text(f'unknown command: {line}'),
                     kind, fold=None, source=out)
            if name in ('/plan', '/todo', '/todos'): self.show_plan = bool(self.agent.plan)
            return None
        got = [self.attach(p) for p in attach_refs(line)]   # `@path` in a prompt attaches it
        if got: self.note('\n'.join(got))
        return run_turn(self, line)

    def on_key(self, k):
        "One keystroke. Returns a coroutine to spawn, `'quit'`, or None."
        if self.mode == 'python' and self.ask is None and not self.buf.text.lstrip().startswith('/'):
            if (k.name == 'tab' and self.complete is None and self.turn is None
                    and self.buf.text): return self.complete_python()
            if (k.name == 'enter' and self.menu is None and self.buf.text.strip()
                    and not self.buf.text.endswith('\n\n')):
                from ramabana.pyrepl import _syntax_note, code_state
                state = code_state(self.buf.text)
                if state == 'incomplete':
                    self.buf.insert('\n')
                    return self.paint()
                if state == 'invalid':
                    self.say(Text(f'incomplete or invalid: {_syntax_note(self.buf.text)}'), 'error', fold=None)
                    return self.paint()
            if k.name == 'ctrl+c' and self.turn is not None:
                self.comp.spawn(self.kernel.interrupt(), name='interrupt')
                self.buf.clear()
                return self.paint()
        self.desc = []
        if self.menu is not None:
            done, choice = self.menu.choose(k.name)
            if not done: return self.paint()
            prompt, self.menu_prompt, self.menu = self.menu_prompt, '', None
            if choice is None or choice.suffix is None:
                self.buf.text, self.buf.cursor = prompt, len(prompt)
                return self.paint()
            self.say(Text(prompt), 'user')
            self.say(Text(f'{choice.label} -- {choice.note}'), 'note')
            self.paint()
            return run_turn(self, prompt + choice.suffix)
        if self.complete is not None and k.name in ('tab', 'shift+tab', 'up', 'down'):
            if k.name in ('tab', 'down'): self.complete.cycle(1)
            else: self.complete.cycle(-1)
            return self.paint()
        if self.complete is not None and k.name == 'escape':
            self.complete = None
            return self.paint()
        if k.name == 'ctrl+t':
            self.show_plan = not self.show_plan
            return self.paint()
        if self.ask is not None:
            bare = not self.buf.text.strip()
            if bare and k.name in ('y', 'Y'):     self.answer(True)
            elif bare and k.name in ('n', 'N'):   self.answer(False)
            elif bare and k.name in ('a', 'A'):   self.answer(True, session=True)
            elif k.name == 'enter':               self.answer(False)   # a typed reason is a refusal
            elif k.name == 'ctrl+y':              self.answer(True)    # ...unless approved with it as guidance
            elif k.name == 'ctrl+c':              self.answer(False)     # stopping the turn refuses what it was waiting on
            else: self.buf.handle(k)
            return self.paint()
        if k.name == 'ctrl+d' and not self.buf.text: return 'quit'
        if k.name == 'ctrl+c':
            self.buf.clear()
            # One implementation of stopping, in `stop`: this branch and the `on_key` that wraps it
            # each grew their own, and the seal that keeps late text below the note reached only one.
            if self.turn is not None: return self.stop()
            return self.paint()
        if k.name == 'enter':
            coro = self.submit()
            self.paint()
            return coro
        if k.name == 'ctrl+v':
            self.note(self.attach_clipboard())
            return self.paint()
        if k.name == 'ctrl+o':
            self.fold_work()
            return self.paint()
        if k.name.startswith('alt+') and k.name[4:].isdigit() and k.name[4:] != '0':
            self.drill(int(k.name[4:]))
            return self.paint()
        if k.name == 'tab' and self.buf.text.startswith('/') and self.complete is None:
            self._refresh_complete()
            if self.complete is not None:
                if not self.complete.insert_common(): self.complete.cycle(1)
                return self.paint()
        self.buf.handle(k)
        if self.buf.text.startswith('/'): self._refresh_complete()
        else:
            self.complete, self.desc = None, []
            self._suggest_soon()
        self.paint()

    def reply(self, text):
        "How a model reply renders. Plain here; the Markdown layer below patches it."
        return Text(text)

    def flush_stream(self):
        """Render the open segment now, whatever `STREAM_EVERY` would have said. Every boundary calls it.

        A no-op when the drawing already matches the model, which is what lets `animate` call it on
        every frame: without a timer the last chunk before a stall stayed invisible for as long as
        the model paused, and with one but no `_rendered` check it would re-render the whole reply
        ten times a second for nothing.
        """
        if self._seg_blk is None or not self._seg or self._seg == self._rendered: return
        self.comp.set_body(self._seg_blk, self.reply(self._seg), source=self._seg)
        self.comp.refresh_block(self._seg_blk)
        self._rendered, self._painted_at = self._seg, time.monotonic()
        self.touch()

    def stream(self, blk, chunk):
        "Grow one segment of the turn's timeline, opening a block where a call ended the last one."
        if blk is None: self._seg_blk = None
        if self._seg_blk is None: self._seg = ''
        self._seg += chunk
        self._reply += chunk
        if self._seg_blk is None:
            self._seg_blk = self.say(self.reply(self._seg), 'reply', fold=None, source=self._seg)
            self._rendered, self._painted_at = self._seg, time.monotonic()
        else:
            self._seg_blk.source = self._seg    # cheap, and it is what `y` and `/copy` read
            if time.monotonic() - self._painted_at >= STREAM_EVERY: self.flush_stream()
        return self._seg_blk

In [ ]:
#| export
@patch
def show_media(self: Ui, media, session=''):
    "Save the pictures a turn generated and name them in the transcript."
    for m in media or []:
        try: p = save_media(m, session or getattr(self.agent, 'session_dir', '') or '.')
        except Exception as e:
            self.say(Text(f'could not save generated media ({agent_err(e)})'), 'error')
            continue
        self.say(Text(media_line(p)), 'note')

In [ ]:
class _Comp:
    cols = 80
    class tty:
        @staticmethod
        def write(s): pass
class _Say:
    def __init__(self): self.said, self.comp, self.agent = [], _Comp(), None
    def say(self, body, kind='reply', **kw): self.said.append((str(body), kind))

u = _Say(); Ui.show_media(u, [])
test_eq(u.said, [])

import tempfile
with tempfile.TemporaryDirectory() as d:
    u = _Say()
    Ui.show_media(u, [{'mime': 'image/png', 'data': b'\x89PNG\r\n\x1a\nx'}], session=d)
    assert u.said and (Path(d)/'media'/'image-1.png').exists()

In [ ]:
#| export
@patch
def tail(self:Ui):
    "The live-tail description shared with Teleprint's transcript view."
    rows = [self.status()]
    if self.hint: rows.append(Text(' ' + self.hint, style=GRUVBOX['gray']))
    chips = self.attach_row()
    if chips is not None: rows.append(chips)
    rows += self.working()     # last, so "where is it now" is always the row above what you type
    rows.append(self.prompt())
    before = Text(self._prefixed(self.buf.text[:self.buf.cursor]))
    rendered = self.comp.console.render_lines(before, pad=False)
    cursor = (len(rows) - 1, len(rendered) - 1, sum(s.cell_length for s in rendered[-1]))
    return rows, cursor

@patch
def recall(self:Ui, step):
    "Move through submitted prompts, preserving the draft beyond the newest entry."
    if not self.history: return False
    if self.history_at == len(self.history): self.draft = self.buf.text
    self.history_at = max(0, min(len(self.history), self.history_at + step))
    text = self.draft if self.history_at == len(self.history) else self.history[self.history_at]
    self.buf.text, self.buf.cursor = text, len(text)
    return True

_core_submit = Ui.submit

@patch
def remember(self:Ui, line):
    "Keep `line` for Ctrl-P/Ctrl-N recall."
    if line and (not self.history or self.history[-1] != line): self.history.append(line)
    self.history_at, self.draft = len(self.history), ''

@patch
def submit(self:Ui):
    "Submit the line and remember it for Ctrl-P/Ctrl-N recall."
    self.remember(self.buf.text.strip())
    return _core_submit(self)

_core_on_key = Ui.on_key

@patch
def enter_transcript(self:Ui):
    "Open the browsing view, borrowing the mouse for exactly as long as it is up."
    self.flush_stream()   # it renders bodies, so a throttled segment must land before it reads them
    self.comp.tty.write(MOUSE_ON)
    self.transcript.enter()

@patch
def leave_transcript(self:Ui):
    "Close the browsing view and give the mouse back, unless `/mouse` says it is this surface's."
    self.transcript.leave()
    if not self.mouse: self.comp.tty.write(MOUSE_OFF)

@patch
def set_mouse(self:Ui, want=''):
    "Take the mouse on the main screen, or give it back to the terminal. Off by default."
    want = (want or '').strip().lower()
    if want not in ('', 'on', 'off', 'toggle', 'yes', 'no'): return 'usage: /mouse [on|off]'
    self.mouse = not self.mouse if want in ('', 'toggle') else want in ('on', 'yes')
    if not self.transcript.active: self.comp.tty.write(MOUSE_ON if self.mouse else MOUSE_OFF)
    return ('mouse on: click a block to fold it, wheel up to browse; shift-drag still selects'
            if self.mouse else 'mouse off: selection belongs to the terminal again')

@patch
def on_mouse(self:Ui, ev):
    "Who owns the mouse: the browsing view while it is up, this surface only when `/mouse` says so."
    if self.transcript.active: return self.transcript.on_mouse(ev)
    if not self.mouse: return True
    if ev.press and ev.btn == 64:
        self.enter_transcript()
        return True
    return ev.press and ev.btn == 65

@patch
def on_key(self:Ui, k):
    "Give transcript navigation priority. Keep prompt recall on Ctrl-P/Ctrl-N."
    view = self.transcript
    if view.active:
        if view.on_key(k): return None
        if k.name == 'escape':
            self.leave_transcript()
            return self.paint()
        if k.name == 'enter':
            if not view.composing: return None
            out = self.submit()
            self.leave_transcript()
            self.paint()
            return out
        out = _core_on_key(self, k)
        view.rebuild(bottom=view.follow)
        return out
    if (k.name == 'ctrl+r' or (k.name in ('up', 'down') and not self.buf.text)) and self.ask is None:
        self.enter_transcript()
        if k.name in ('up', 'down'): view.on_key(k)
        return None
    if k.name in ('ctrl+p', 'ctrl+n'):
        self.recall(-1 if k.name == 'ctrl+p' else 1)
        return self.paint()
    return _core_on_key(self, k)

@patch
def paste(self:Ui, text):
    "Insert pasted text, or attach the files when the paste is nothing but paths."
    found = media_paths(text)
    if found:
        for p in found: self.attach(p)
    else: self.buf.insert(text)
    if self.transcript.active:
        if not found: self.transcript.composing = True
        return self.transcript.rebuild(bottom=self.transcript.follow)
    return self.paint()

@patch
def reply(self:Ui, text):
    "A model reply as Markdown, its code highlighted in whatever style the active palette asks for."
    return Markdown(text, code_theme=code_theme(), style=GRUVBOX['fg1'])

### Python mode

One program, one prompt, and two things a line can mean. `/python` starts a kernel of the user's own and takes the line for Python. `/agent` hands it back and leaves the kernel running. Only `/quit` and the process ending shut it down. There is no second surface: the transcript, folding, approvals, attachments and slash commands are the same ones, and what changes is what `enter` does with the line.

The agent moves with the mode. `enter_python` swaps the host for a `DhrishtiHost` over the new kernel and calls `Agent.refresh`, which drops the skill, registry and tool caches and re-briefs the running turn backend in place. The agent's Python becomes Dhrishti's protected overlay mid-session: it reads the namespace being typed into, writes into a layer of its own, and cannot rebind a name the owner made.

Everything a Python prompt has to judge. The kernel, the overlay host, whether a line compiles, how to colour it. Lives in [pyrepl](11_pyrepl.ipynb), imported on demand so a plain `ramabana` needs no kernel. The surface tests that need a live one are there too.

In [ ]:
#| export
PYREPL_MODULES = ('jupyter_client', 'dhrishti')

@patch
def log_cell(self:Ui, source, outputs=None, cell_type='code'):
    "Write one cell to the session notebook, when the host keeps one."
    log = getattr(self.agent.host, 'log_cell', None)   # only a `DhrishtiHost` has one
    if log is not None: log(source, outputs, cell_type)

@patch
def use_host(self:Ui, host):
    "Move the agent onto `host`, re-briefing the running turn backend with its tools."
    self.agent.host = host
    if self.agent.approvals is not None: self.agent.approvals.host = host   # `create_file` previews through it
    self.agent.refresh()

@patch
async def enter_python(self:Ui):
    "Take the line for Python, starting the kernel the first time and pointing the agent at its overlay."
    try:
        if self.kernel is None:
            if self.attached:
                self.attached = ''   # leaving it: their kernel keeps running, we just stop pointing at it
                self.note('left the attached session; starting a kernel of your own')
            if missing := [m for m in PYREPL_MODULES if find_spec(m) is None]:
                return self.note(f"python mode needs {' and '.join(missing)}: pip install 'ramabana[pyrepl]'", 'error')
            from ramabana.pyrepl import DhrishtiHost, Kernel
            self.note('starting a kernel')
            try: self.kernel = await Kernel(self.agent.host.roots[0]).start()
            except Exception as e:
                self.kernel = None
                return self.note(f'no kernel: {agent_err(e)}', 'error')
            old = self.agent.host
            self.use_host(DhrishtiHost(old.roots, self.kernel.base, approvals=old.approvals,
                                       web=old.web, read_outside=old.read_outside))
        self.mode = 'python'
        return self.note('python mode · /agent hands the line back')
    finally:
        self.turn = None
        self.paint()

@patch
async def attach_session(self:Ui, name):
    "Point the agent's Python at a dhrishti session someone else owns. We start nothing and own no prompt."
    from ramabana.pyrepl import DhrishtiHost, find_session
    old, base = self.agent.host, find_session(name)
    self.use_host(DhrishtiHost(old.roots, base, approvals=old.approvals, web=old.web,
                               read_outside=old.read_outside))
    self.attached = base
    return base

@patch
async def run_code(self:Ui, code):
    "Run one owner-typed Python cell, streaming its outputs into blocks."
    result = None
    try:
        result = await self.kernel.execute(code, on_output=self.on_output)
        if not result.ok and not result.outputs: self.say(Text(result.error or 'execution failed'), 'error')
    finally:
        self.turn = None
        self.paint()
    if result is not None: self.log_cell(code, result.outputs)

@patch
def on_output(self:Ui, output):
    "One nbformat output in the block that means it, as the cell produces it."
    from ramabana.pyrepl import _text
    kind = output.get('output_type')
    if kind == 'stream':
        if text := _text(output.get('text')).rstrip('\n'): self.say(Text(text), 'note')
    elif kind in ('execute_result', 'display_data'):
        data = output.get('data') or {}
        if 'text/plain' in data: self.say(Text(_text(data['text/plain'])), 'reply')
        elif 'text/markdown' in data: self.say(Text(_text(data['text/markdown'])), 'reply')
        elif data: self.say(Text('display: ' + ', '.join(data)), 'note')
    elif kind == 'error':
        body = '\n'.join(output.get('traceback') or [])
        self.say(Text.from_ansi(body or f"{output.get('ename')}: {output.get('evalue')}"), 'error')

@patch
async def _complete(self:Ui, insert):
    """List what the kernel would complete. `insert` is tab: typing must not rewrite the buffer.
    `CompletionMenu` owns the span. Tab cycles and shift+tab goes back. `desc` is a separate
    line because cycling writes the highlighted match into the buffer and a type would go in too.
    """
    from ramabana.pyrepl import annotate
    identity = (self.buf.text, self.buf.cursor)   # typing is not gated during the awaits
    def stale(): return (self.buf.text, self.buf.cursor) != identity
    matches, start = await self.kernel.complete(self.buf.text, self.buf.cursor)
    if not matches or stale():
        if not stale(): self.complete, self.desc = None, []
        return
    m = CompletionMenu(self.buf, list(matches), start=start, show=8)
    if insert and m.insert_common(): identity = (self.buf.text, self.buf.cursor)   # ours, not theirs
    if insert and len(matches) == 1: return
    self.complete, self.desc = m, []
    self.paint()
    described = await asyncio.to_thread(self.agent.host.describe)
    if stale() or self.complete is not m: return
    self.desc = [x for x in annotate(matches, described) if ' -> ' in x][:4]

@patch
async def complete_python(self:Ui):
    "Tab: complete as far as the candidates agree, and leave the menu up to cycle."
    try: await self._complete(insert=True)
    finally:
        self.turn = None
        self.paint()

@patch
async def _suggest(self:Ui):
    "The same list, kicked off by typing. Never touches the buffer and never owns `turn`."
    try: await self._complete(insert=False)
    except Exception: self.complete, self.desc = None, []   # a dead kernel must not kill a keystroke
    finally:
        self._suggesting = False
        self.paint()

@patch
async def _promote(self:Ui, name):
    "Adopt one agent variable, off the loop thread: the owner-token call can take a minute."
    from ramabana.pyrepl import promote
    if self.attached:   # their token, their namespace, their surface to adopt from
        return self.note(f'{self.attached} belongs to whoever started it, so promoting is theirs to '
                         'do. The agent works in its own layer here and cannot write into the kernel.')
    self.say(Text(await asyncio.to_thread(promote, self.kernel.base if self.kernel else '', name)),
             'note', fold=None)
    self.turn = None
    self.paint()

In [ ]:
# The swap is the new behaviour: the agent's `run_python` reaches the overlay from that moment,
# and its tool list is rebuilt rather than served from the cache. A fake kernel and a fake overlay,
# so this page needs no dhrishti.
import ramabana.pyrepl as _pyrepl
from ramabana.testing import MemHost

class _Mem(MemHost):
    "A host that records what the agent runs. A swap is visible in what it recorded."
    def __init__(self, *a, **kw):
        super().__init__()
        self.ran, self.web, self.read_outside = [], False, False
    def run_python(self, code): self.ran.append(code); return '(ok)'
    def list_vars(self): return 'mem_var: int = 1'
    def terminal_text(self, n=1): return ''

class _Overlay(_Mem):
    "Stands in for `DhrishtiHost`: the same constructor, and a `base` to prove which host answered."
    def __init__(self, roots, base, **kw): super().__init__(); self.base = base

class _FakeKernel:
    base = 'http://127.0.0.1:65000'
    def __init__(self, cwd='.'): self.cwd = cwd
    async def start(self): return self

def _run_python(a): return next(t for t in a.tools if t.__name__ == 'run_python')

def _run_python(a): return next(t for t in a.tools if t.__name__ == 'run_python')
def _said(u, c): return u.transcript.block_text(list(c.blocks.values())[-1])

swap_tty = EmuTty(72, 14)
swap_comp = Compositor(swap_tty)
swap_comp._register_signals = lambda: None
await swap_comp.start()
swap_agent, _ = fake_agent(host=_Mem(), approvals=Approvals(tools=WRITE_TOOLS, mode='ask'))
swap_ui = Ui(swap_comp, swap_agent)

_run_python(swap_agent)('before = 1')
test_eq(swap_agent.host.ran, ['before = 1'])       # the mem host answers, and the list is cached
tools_before = swap_agent.tools

_reals, PYREPL_MODULES = (_pyrepl.Kernel, _pyrepl.DhrishtiHost), ()   # fakes. This page needs no dhrishti
_pyrepl.Kernel, _pyrepl.DhrishtiHost = _FakeKernel, _Overlay
try: await swap_ui.enter_python()
finally: _pyrepl.Kernel, _pyrepl.DhrishtiHost = _reals

test_eq((swap_ui.mode, swap_ui.kernel.base), ('python', _FakeKernel.base))
test_eq(swap_agent.host.base, _FakeKernel.base)              # the overlay host is the agent's now
test_eq(swap_agent.approvals.host, swap_agent.host)          # ...and the gate previews through it
assert swap_agent.tools is not tools_before                  # rebuilt, not the cached list
_run_python(swap_agent)('after = 2')
test_eq(swap_agent.host.ran, ['after = 2'])                  # the agent's Python reaches the overlay

# Without the extra it refuses by name, rather than failing inside `Kernel.start` where the
# message cannot say what is missing.
swap_ui.kernel, swap_ui.mode = None, 'agent'
PYREPL_MODULES = ('no_such_module',)
try: await swap_ui.enter_python()
finally: PYREPL_MODULES = ()
test_eq((swap_ui.mode, swap_ui.kernel), ('agent', None))
assert 'ramabana[pyrepl]' in _said(swap_ui, swap_comp)

# `--attach` points the agent at a session someone else owns and starts nothing of ours.
_reals = _pyrepl.DhrishtiHost, _pyrepl.find_session
_pyrepl.DhrishtiHost, _pyrepl.find_session = _Overlay, lambda name: f'http://127.0.0.1:9{name}'
try: test_eq(await swap_ui.attach_session('001'), 'http://127.0.0.1:9001')
finally: _pyrepl.DhrishtiHost, _pyrepl.find_session = _reals
test_eq((swap_ui.kernel, swap_ui.mode, swap_ui.attached), (None, 'agent', 'http://127.0.0.1:9001'))

# `/python` from there is not a dead end: it stops pointing at their session and starts one of
# ours. Their kernel keeps running. We only let go of it.
_realk = _pyrepl.Kernel
_pyrepl.Kernel = lambda root: _FakeKernel()
try: await swap_ui.enter_python()
finally: _pyrepl.Kernel = _realk
test_eq((swap_ui.attached, swap_ui.mode), ('', 'python'))
assert swap_ui.kernel is not None
assert any('left the attached session' in swap_ui.transcript.block_text(b)
           for b in swap_comp.blocks.values())      # `_said` is the last block. This is two back
swap_comp.stop()

/Users/71293/code/personal/orgs/ramabana/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Assistant Markdown keeps distinct semantic styles instead of flattening everything to terminal foreground. The model's source remains plain Markdown. Only its rendering gains color. Raw Rich markup cannot escape into the terminal.

Some requests have more than one honest reading, and the expensive part is finding out which one you got. `options_for` decides whether a typed line earns a row of choices, `ChoiceMenu` owns navigating and picking one, and `Ui` decides what the pick *means*. The component is reusable for the next question rather than wired to this one. Each `Option` carries the instruction it appends, because a preference the model may weigh against the rest of its briefing is not a choice the user made.

In [ ]:
menu = ChoiceMenu(*options_for('refactor the runtime'))
test_eq(menu.choose('down'), (False, None))          # navigation is not a decision
test_eq(menu.choose('enter')[1].label, 'plan')
test_eq(ChoiceMenu('x', REFACTOR).choose('s')[1].label, 'scope')   # its own letter picks it
test_eq(ChoiceMenu('x', REFACTOR).choose('escape'), (True, None))
test_eq(options_for('what does the runtime do?'), None)            # an ordinary question gets no row
print('\n'.join(r.plain for r in menu.render()))

 how should I take this?
   a  apply    edit the files, then run the project’s checks
 › p  plan     read only, propose the steps
   s  scope    name what is in and out, then stop
   c  cancel   put the line back to edit


In [ ]:
from rich.console import Console
color_console = Console(width=72, theme=MARKDOWN_THEME, color_system='truecolor')
styled = Markdown('# Heading\n\nUse `search_code` and **verify** it.\n\n```python\nanswer = 42\n```',
                  code_theme='github-dark')
lines = color_console.render_lines(styled, pad=False)
colors = {seg.style.color.triplet for line in lines for seg in line
          if seg.style and seg.style.color and seg.style.color.triplet}
inline = next(seg for line in lines for seg in line if seg.text == 'search_code')
test_eq(inline.style.bgcolor, None)
test_eq(len(colors) >= 3, True)

A headless terminal, a real compositor, and an agent whose model is a fake: the whole surface, in a notebook.

In [ ]:
tty = EmuTty(72, 14)
comp = Compositor(tty)
comp._register_signals = lambda: None  # nbdev executes this async cell on a worker thread
await comp.start()
agent, be = fake_agent(replies=['`threshold` is in `ramabana/runtime.py`, and it caps the reserve.'], local_multimodal=True)
ui = Ui(comp, agent)
comp.on_key = ui.on_key          # a coroutine returned by a handler is spawned by the dispatcher
ui.paint()
print(tty.term.text())

RAMABANA 0.1.19  fake  ● idle  26 tools · 23 skills · 0% ctx
▌


In [ ]:
ui.buf.insert('refactor the runtime')
test_eq(ui.submit(), None)                    # a row, not a turn
ui.paint()
assert 'a  apply' in tty.term.text() and 'p  plan' in tty.term.text()
test_eq(ui.menu.choose('escape'), (True, None))
ui.menu, ui.menu_prompt = None, ''
ui.buf.clear()
ui.paint()

Cancelling hands the line back to the composer rather than throwing it away: the user typed it, and the row was the harness's question, not theirs.

In [ ]:
ui.buf.insert('restructure the compactor')
ui.submit()
ui.on_key(Key('escape'))
test_eq(ui.buf.text, 'restructure the compactor')   # given back, still editable
ui.buf.clear()
ui.paint()

That is the resting state. The status bar says what is loaded and how full it is, and the cursor sits on the prompt. Everything above the last two rows is the transcript.

In [ ]:
test_eq(tty.term.text().splitlines()[-1], '▌')
[l for l in tty.term.text().splitlines() if 'tools' in l]

['RAMABANA 0.1.19  fake  ● idle  26 tools · 23 skills · 0% ctx']

Typing goes through the real key parser: these are the bytes a terminal sends, not synthesised `Key` objects.

In [ ]:
comp.on_bytes(b'where is the threshold?')
tty.term.text().splitlines()[-1]

'▌ where is the threshold?'

The working marker advances independently of model output. A silent network wait still looks alive. When no turn is running it returns to a stable ready dot.

In [ ]:
resting_status = ui.status().plain
assert VERSION in resting_status   # a stale session says so on the bar
ui.frame += 1
# At rest the arrow does not move, and the bar does not claim to be working.
test_eq(ui.status().plain, resting_status)
assert ' working' not in resting_status and 'WORKING' not in resting_status

ui.turn = object()
first = ui.status().plain
ui.frame += 1
second = ui.status().plain
ui.turn = None
test_eq(' working' in first and ' working' in second, True)
test_ne(first, second)                        # under way, the arrow has flown on by the next frame
test_eq(cell_len(first), cell_len(second))    # and the bar never changes width while it does


Enter prints the question as a block and spawns the turn. Awaiting a moment lets the model thread run. In the real app that wait is the event loop, which is why nothing here blocks it.

In [ ]:
comp.on_bytes(b'\r')
await asyncio.sleep(0.4)
print(tty.term.text())


▌ where is the threshold?
│ 🔍 Search where is the threshold? … (+1 lines)
  threshold is in ramabana/runtime.py, and it caps the reserve.
RAMABANA 0.1.19  fake  ● ready  26 tools · 23 skills · 2% ctx · 15 tok ·
in 10 · out 5 · model
▌


In [ ]:
#| eval: false
ui.buf.text, ui.buf.cursor = 'x' * 80, 80
ui.paint()
test_eq(comp._cursor[1], len(ui.buf.text) % tty.size[0])
test_eq(0 <= comp._cursor[0] < comp.rows, True)
ui.buf.text, ui.buf.cursor = 'top\nbottom', len('top\nbottom')
ui.paint()
test_eq(comp._cursor[1], len('bottom'))
test_eq(0 <= comp._cursor[0] < comp.rows, True)
ui.buf.clear()
ui.paint()

Prompt history uses Ctrl-P/Ctrl-N, preserving a draft. With an empty composer, Up or Down enters Teleprint's transcript browser instead.

In [ ]:
ui.buf.insert('first prompt')
ui.submit()
ui.buf.insert('second prompt')
ui.submit()
ui.buf.insert('unfinished draft')
ui.on_key(Key('ctrl+p'))
test_eq(ui.buf.text, 'second prompt')
ui.on_key(Key('ctrl+p'))
test_eq(ui.buf.text, 'first prompt')
ui.on_key(Key('ctrl+n'))
ui.on_key(Key('ctrl+n'))
test_eq(ui.buf.text, 'unfinished draft')
ui.buf.clear()
ui.on_key(Key('up'))
test_eq(ui.transcript.active, True)
ui.on_key(Key('g'))
test_eq(ui.transcript.top, 0)
ui.on_key(Key('escape'))
test_eq(ui.transcript.active, False)

/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/ipykernel_68843/1243914784.py:2: RuntimeWarning: coroutine 'run_turn' was never awaited
  ui.submit()
/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/ipykernel_68843/1243914784.py:4: RuntimeWarning: coroutine 'run_turn' was never awaited
  ui.submit()


The transcript now holds the question, the tool calls the agent made on the way, and the reply. Each one a block with its own gutter.

In [ ]:
[(b.tag, b.height) for b in comp.blocks.values()]

[('user', 1), ('tool', 2), ('reply', 1), ('user', 1), ('user', 1)]

In [ ]:
#| export
@patch
def _close_seg(self:Ui): self.flush_stream()

@patch
def _act(self:Ui, act):
    if act.parent_action_id and act.parent_action_id in self.acts: return self._nest(act)
    self.by_id[act.id] = act
    blk = self.acts.get(act.id)
    if blk is None:
        self._close_seg()
        if self._reply: self._seg += '\n'; self._reply += '\n'
        line = Text(act.line(), style=self._act_style(act))
        self.acts[act.id] = self.say(line, 'tool', source=act.line())
        return self.paint()
    if act.id in self.kids: self._paint_group(act.id)
    else:
        line = Text(act.line(), style=self._act_style(act))
        src = act.line() if not act.detail else f'{act.line()}\n{act.detail}'
        body = [line] if not act.detail else [line, Text(act.detail, style=GRUVBOX['gray'])]
        self.comp.set_body(blk, *body, source=src)
        blk.collapsed = self._folded(act, blk)
        self.comp.refresh_block(blk)
        if act.done: self.by_id.pop(act.id, None)
    self.touch()
    now = time.monotonic()
    if act.done or now - self._acted_at >= ACT_EVERY:
        self._acted_at = now
        self.paint()


In [ ]:
test_eq(agent.calls[0][0], 'search_code')       # the preflight ran, and the feed recorded it
test_eq(any('threshold' in str(b.body) for b in comp.blocks.values()), True)
be.sent and len(be.sent)

1

## Folding

A block taller than `FOLD` is born folded. A four-hundred-line reply costs one row until it is asked for. Ctrl-O opens the whole turn's working, every step and every call, and shuts it again. After a long turn the newest block is the least interesting one to reach for. The main screen leaves the mouse to the terminal. Selection and copy work there as in any scrollback. In the transcript view any block toggles, under the block cursor or under a click.

In [ ]:
long = ui.say(Text('\n'.join(f'line {i}' for i in range(40))), 'tool')
long.collapsed, long.height

(True, 40)

In [ ]:
comp.toggle(long)
test_eq(long.collapsed, False)
comp.toggle(long)
long.collapsed

True

A one-line block has nothing to hide, and toggling it is a no-op rather than an error.

In [ ]:
short = ui.say(Text('one line'), 'tool')
comp.toggle(short), short.collapsed

(None, False)

A tool call folds to its own summary line rather than to `FOLD`, because the summary is the message. A turn of thirty calls reads as thirty lines, and any one output is one Ctrl-O away.

`FOLD` cannot state that policy on its own. A tool block is printed when the call starts, one hourglass row tall, and its result is set into the same block when the call returns. The born-folded decision in `print_block` is made against a height of 1 and never revisited, and every result printed in full however long it was. `_act` therefore decides the fold at the moment the result lands, which is the first moment there is one.

In [ ]:
from ramabana.agent import Act

step = Act(tool='view_file', args={'path': 'runtime.py'}, summary='view runtime.py')
ui.on_act(step)                                        # running: an hourglass, nothing to fold
test_eq((ui.acts[step.id].height, ui.acts[step.id].collapsed), (1, False))

ui.on_act(step.finish('\n'.join(f'line {i}' for i in range(400))))
folded = ui.acts[step.id]
test_eq((folded.height, folded.collapsed), (401, True))  # the result is there, and it is one row
test_eq(len(comp._block_rows(folded)), 1)
assert step.summary in ui.transcript.block_text(folded) # ...the row being the step's own summary

comp.toggle(folded)                                     # drilling in stays one keystroke away
test_eq((folded.collapsed, len(comp._block_rows(folded))), (False, 401))
comp.toggle(folded)

The activity feed is the record of a tool call. Prose streamed before that call is interim narration, so the tool boundary removes it. The completed trace stays on screen and the reply block begins with prose after the final call.

The reply is tagged `reply` while it is open. `/copy` reaches the final answer. Tool calls are the foldable trace above it.

In [ ]:
tl_tty = EmuTty(72, 20)
tl_comp = Compositor(tl_tty)
tl_comp._register_signals = lambda: None
await tl_comp.start()
tl_agent, _ = fake_agent()
tl_ui = Ui(tl_comp, tl_agent)
tl_acts = tl_agent.activity          # `Ui` wired itself to `activity.on_change` in its constructor

# two paragraphs, so the tool result is taller than one row and therefore has something to fold
seg = tl_ui.stream(None, 'Looking for it.\n\nTwo places, probably.\n')
call = tl_acts.start('search_code', {'query': 'threshold'})
tl_acts.finish(call, 'runtime.py:88\nagent.py:12')
seg = tl_ui.stream(seg, '## Answer\n\nBoth of them.\n')
tl_ui.flush_stream()
tl_ui._seg_blk = None   # the turn ends here, as `run_turn`'s finally does: no later call may end
                        # the answer's segment, and so retag the answer as working

timeline = list(tl_comp.blocks.values())
test_eq([b.tag for b in timeline], ['tool', 'reply'])       # trace before the merged reply
test_eq([b.collapsed for b in timeline], [True, False])       # trace folded, reply open
test_eq([len(tl_comp._block_rows(b)) for b in timeline[:1]], [1])
# one reply, all of the turn's prose in it, exactly as the `note` case below asserts
test_eq(tl_ui.transcript.block_text(timeline[-1]), '''Looking for it.

Two places, probably.

## Answer

Both of them.
''')

status_tty = EmuTty(72, 10)
status_comp = Compositor(status_tty)
status_comp._register_signals = lambda: None
await status_comp.start()
status_agent, _ = fake_agent()
status_ui = Ui(status_comp, status_agent)
status_seg = status_ui.stream(None, 'Waiting for approval.\n')
status_ui.note('approved')
status_ui.stream(status_seg, 'Done.\n')
status_blocks = list(status_comp.blocks.values())
test_eq([b.tag for b in status_blocks], ['note', 'reply'])
test_eq(status_ui.transcript.block_text(status_blocks[-1]), 'Waiting for approval.\nDone.\n')

In [ ]:
# `/copy` reaches the final reply. `/copy turn` reaches the trace and reply.
answer = timeline[-1]
test_eq(tl_ui.transcript.block_text(answer),
        'Looking for it.\n\nTwo places, probably.\n\n## Answer\n\nBoth of them.\n')
assert 'copied' in tl_ui.copy_last('reply')

# Ctrl-O reaches the turn, not the newest block. At rest everything is shut, so the first press opens.
test_eq(tl_ui.fold_work(), False)
test_eq([b.collapsed for b in timeline], [False, False])
test_eq(tl_ui.fold_work(), True)
test_eq([b.collapsed for b in timeline], [True, False])       # the reply is never `fold_work` work

# the plan is one block that gets rewritten, not one copy per todo change: `on_plan` fires after
# every todo, and a `say` per fire stacked 0/5, 1/5, 2/5 down the transcript, each frozen at its count
class _FakePlan:
    def __init__(self): self.n = 0
    def md(self):
        rows = [f"{'[x]' if i < self.n else '[ ]'} step {i+1}" for i in range(3)]
        return f"**Repair**  \u00b7  {self.n}/3 done\n" + '\n'.join(rows)

plan_tty = EmuTty(72, 20)
plan_comp = Compositor(plan_tty)
plan_comp._register_signals = lambda: None
await plan_comp.start()
plan_agent, _ = fake_agent()
plan_ui = Ui(plan_comp, plan_agent)
plan_ui._plan_blk = None
_p, _blk = _FakePlan(), None
for _i in range(3):
    _p.n = _i
    plan_ui.on_plan(_p)
    _blk = plan_ui.stream(_blk, f'step {_i+1}\n')
plan_ui.flush_stream()
_plans = [b for b in plan_comp.blocks.values() if b.tag == 'plan']
test_eq(len(_plans), 1)                                    # one block, however many todos moved
assert '2/3 done' in plan_ui.transcript.block_text(_plans[0])   # and it holds the latest count

# the checklist is themed: marks in one colour, step text in another, ids and notes receding to gray
_md = ('**Repair**  \u00b7  1/2 done\n'
       '[x] `aaa` did the thing  -- with a note\n'
       '[ ] `bbb` still to do')
_t = plan_text(_md)
test_eq(_t.plain.splitlines()[1], '[x] aaa did the thing  -- with a note')   # plain text unchanged
_styles = {_md_line: None for _md_line in ()}
_mark = [sp for sp in _t.spans if _t.plain[sp.start:sp.end] in ('[x]', '[ ]')]
test_eq(len(_mark), 2)                                    # both marks styled
test_eq(len({sp.style for sp in _mark}), 1)               # ...and identically: one colour for marks
_step = next(sp for sp in _t.spans if _t.plain[sp.start:sp.end].strip() == 'did the thing')
assert _step.style != _mark[0].style                      # step text is a different colour
# `/theme` restyles a plan painted under the old one, since the palette is read per call
set_theme('dark');  _d = plan_text('[x] `a` x').spans[0].style
set_theme('light'); _l = plan_text('[x] `a` x').spans[0].style
assert _d != _l
set_theme('dark')


'dark'

In [ ]:
# The footer is tail rather than transcript: it says where the model is at while a turn runs, and
# because tail rows never ink it leaves nothing behind to scroll past afterwards.
tl_ui._turn_at, tl_ui.turn = time.monotonic(), 'a turn'
footer = [r.plain for r in tl_ui.working()]
test_eq(footer[0], f'1 {call.line()}')          # the digit alt+1 would use, in front of what it reaches
# newest first, and the answer is not among them: `reply` blocks are never working
test_eq(tl_ui.drillable(), [tl_ui.acts[call.id]])
assert footer[-1].startswith('  step 1 ·'), footer
tl_ui.turn = None
test_eq(tl_ui.working(), [])

A delegation is one entry, not a scattering. Its sub-agent runs on the caller's recorded tool wrappers, so its calls reach the same `Activity`. What tells them apart is `parent_action_id`, which `Agent` now sets from a per-thread stack of the delegate calls in flight. Three sub-agents' searches used to arrive as siblings of the caller's own, with nothing saying whose they were. That is most of why a delegating turn read as the same search over and over.

A finished call folds to its summary. A failed one never does, and neither does one still running. An error you have to expand, and progress you cannot see, are the two things folding must not take away.

In [ ]:
parent = tl_acts.start('delegate_search', {'question': 'who reads the threshold?'})
for q in ('threshold callers', 'reserve usage'):
    tl_acts.finish(tl_acts.start('search_code', {'query': q}, parent_action_id=parent.id), 'hits')
tl_acts.finish(parent, 'Three callers, all in runtime.py.')

group = tl_ui.acts[parent.id]
test_eq(len(tl_comp.blocks), 3)                      # one block for the delegate, not three
test_eq((group.collapsed, len(tl_comp._block_rows(group))), (True, 1))
assert '2 calls' in tl_comp._ansi(tl_comp._block_rows(group)[0][1])
assert all(k.line() in tl_ui.transcript.block_text(group) for k in tl_ui.kids[parent.id])

failed = tl_acts.start('run_shell', {'command': 'pytest'})
tl_acts.finish(failed, 'FAILED test_reserve\nassert 0.9 <= 0.8', ok=False)
test_eq(tl_ui.acts[failed.id].collapsed, False)      # a failure is never folded out of sight

In [ ]:
# alt+1..9 drills into one entry, counting back from the newest, and the footer numbers the same
# blocks it reaches. Teleprint's own alt-digit numbering wants a three-glyph gutter; these are two.
test_eq([b.tag for b in tl_ui.drillable()], ['tool', 'tool', 'tool'])
test_eq(tl_ui.drill(2), True)                        # the delegate group, one back from the failure
test_eq(group.collapsed, False)
test_eq(tl_ui.drill(2), True)
test_eq((group.collapsed, tl_ui.drill(99)), (True, False))

# A tool boundary keeps the prior reply: a turn's prose is one merged block, so `/copy` and the
# notebook log get every word of it, not only what was said after the last call.
assert 'copied' in tl_ui.copy_last('turn')
assert 'Looking for it.' in tl_ui._reply and 'Both of them.' in tl_ui._reply

# The mouse is the terminal's until asked for, and the browsing view no longer hands back what it
# was not lent: leaving it with `/mouse` on keeps reporting on.
test_eq(tl_ui.mouse, False)
assert 'mouse on' in tl_ui.set_mouse('on') and tl_ui.mouse
tl_ui.leave_transcript()
test_eq(tl_ui.mouse, True)
assert 'mouse off' in tl_ui.set_mouse('off')

## Copying and browsing

Two ways out of the transcript, and neither of them is a private clipboard. The main screen leaves the mouse to the terminal. Selecting and copying there work as they do in any other scrollback. The transcript view borrows the mouse only for as long as it is up. `/copy` and the view's `y` both go out over OSC 52, which survives ssh and tmux.

What they copy is the block's own text. The Markdown the model wrote, the arguments a tool was given. Not the rendering of it. A reply scraped back off the screen returns wrapped to the terminal's width, indented by its gutter, and stripped of the fences that made its code paste-able. Every block states its source when it is printed.

In [ ]:
md = 'Here is the fix.\n\n```python\nthreshold = 42\n```\n\nCall it from `runtime.py`.'
rblk = ui.stream(None, md[:20])
rblk = ui.stream(rblk, md[20:])
test_eq(ui.transcript.block_text(rblk), md)          # what `y` and `/copy` yield: the model's Markdown
rendered = '\n'.join(''.join(s.text for s in l) for l in comp._content_lines(rblk))
assert '```' not in rendered                          # the fence is gone from the rendering...
assert '```python' in ui.transcript.block_text(rblk)  # ...and intact in what copy yields

In [ ]:
u = ui.say(Text('where is the threshold?'), 'user')
test_eq(ui.transcript.block_text(u), 'where is the threshold?')
wrapped = ui.say('x' * 200, 'note')                  # long enough to wrap over three rows
test_eq(ui.transcript.block_text(wrapped), 'x' * 200)
test_eq(wrapped.height > 1, True)
ui.copy_last('reply')

'copied 74 chars of the last reply'

In [ ]:
ui.enter_transcript()
test_eq((ui.transcript.active, ui.transcript.follow), (True, True))
ui.say(Text('search_code("threshold") landed while browsing'), 'tool')
assert 'landed while browsing' in tty.term.text()   # a new block never waits on the rate limit
sblk = ui.stream(None, 'and a reply streamed ')
sblk = ui.stream(sblk, 'in behind it')
ui.touch(now=True)                                  # ...and the rate limit never loses the last chunk
assert 'in behind it' in tty.term.text()
ui.transcript.move(-1)
test_eq(ui.transcript.follow, False)                # navigating unpins it, as `less` does
ui.leave_transcript()
test_eq((ui.transcript.active, comp.paused), (False, False))

In [ ]:
test_eq(ui.attach(shot), 'attached image shot.png (72B)')   # 8 bytes of PNG signature plus 64
test_eq(ui.attach(shot), 'shot.png is already attached')
ui.attach(snd_path)
test_eq([a.name for a in ui.attachments], ['shot.png', 'note.wav'])
assert 'shot.png' in ui.attach_row().plain and '/detach' in ui.attach_row().plain
test_eq(ui.detach('2'), 'dropped note.wav')
test_eq(ui.detach(), 'dropped 1 attachment')
test_eq(ui.detach(), 'nothing is attached')
test_eq(ui.attach(media_dir / 'gone.png'), 'cannot attach gone.png: no such file')
test_eq(ui.attach_row(), None)

In [ ]:
ui.paste(str(shot))
test_eq([a.name for a in ui.attachments], ['shot.png'])
test_eq(ui.buf.text, '')                              # the path attached instead of being typed
ui.paste('and this is ordinary text')
test_eq(ui.buf.text, 'and this is ordinary text')
long_text = 'copied text ' * 100 + 'image.png'
ui.paste(long_text)
test_eq(ui.buf.text, 'and this is ordinary text' + long_text)
ui.buf.clear()

In [ ]:
ui.buf.insert('what is in this picture?')
await ui.submit()
test_eq(ui.attachments, [])                    # the prompt that named them is the one that carries them

## Approvals in a terminal

The gate blocks the model's worker thread until a person answers. The request arrives on that thread and the answer comes from the loop. The input line becomes the answer: type a reason, then `y`, `n`, or `a` for the rest of the session.

A write, asked for from another thread. Exactly as a tool call does it:

In [ ]:
tty2 = EmuTty(72, 12)
comp2 = Compositor(tty2)
comp2._register_signals = lambda: None  # headless notebook test. No process signals to own
await comp2.start()
gated, _ = fake_agent(replies=['done'])
gated.approvals = Approvals(tools=WRITE_TOOLS, host=gated.host)
ui2 = Ui(comp2, gated)
comp2.on_key = ui2.on_key
ui2.paint()
threading.Thread(target=lambda: gated.approvals.request(
    'create_file', {'path': '/proj/notes.md', 'text': '# notes\n'}), daemon=True).start()
await asyncio.sleep(0.2)
print(tty2.term.text())

◆ create_file → /proj/notes.md

  /proj/notes.md  (new file, 8 chars)

  # notes

RAMABANA 0.1.19  fake  ● idle  26 tools · 23 skills · 0% ctx
approve? [y/n/a, or a reason + enter]


The preview is what the person reads: the path, whether it overwrites, and the head of what would be written. The prompt has become the approval question.

In [ ]:
test_eq(ui2.ask.tool, 'create_file')
tty2.term.text().splitlines()[-1]

'approve? [y/n/a, or a reason + enter]'

The question owns the line whatever the mode is. It is asked here from python mode: a reason is a sentence, and compiling it would leave the gate unanswered.

`y`, `n` and `a` answer only while nothing has been typed. A reason like "put it in docs/ instead" contains all three letters. Once there is text the letters are text and `enter` is the answer. Typing a reason and pressing it refuses *with* that reason, which is the whole point of the gate: something the model can act on rather than the word "denied".

In [ ]:
ui2.mode = 'python'                       # a reason is not code, even where a line would be
comp2.on_bytes(b'put it in docs/ instead')
tty2.term.text().splitlines()[-1]

'approve? [y/n/a, or a reason + enter] put it in docs/ instead'

In [ ]:
comp2.on_bytes(b'\r')
await asyncio.sleep(0.2)
ui2.mode = 'agent'; ui2.paint()
ui2.ask, gated.approvals.history[-1].reply()

(None, 'Denied by human operator. Reason given: put it in docs/ instead')

Prompt history belongs to the composer, while transcript history belongs to Teleprint's block model. Up and Down recall submitted prompts only when `Buffer` cannot move within a multiline prompt. Ctrl-R enters Teleprint's transcript view, where old folded blocks can be searched, expanded, and copied without creating a second history.

In [ ]:
# Helpers are defined beside `Ui`, before the executable surface examples.

In [ ]:
test_eq(bool(gated.approvals.history[-1]), False)
test_eq(tty2.term.text().splitlines()[-1], '▌')      # the prompt is back
[b.tag for b in comp2.blocks.values()][-2:]

['ask', 'note']

`a` is the deliberate bulk answer: it approves this request and switches the policy to `auto` for the rest of the session, which is a thing a person does on purpose rather than by holding down return.

In [ ]:
threading.Thread(target=lambda: gated.approvals.request('create_file', {'path': '/proj/b.md'}), daemon=True).start()
await asyncio.sleep(0.2)
comp2.on_bytes(b'a')
await asyncio.sleep(0.2)
gated.approvals.mode, gated.approvals.history[-1].note

('auto', 'approved for the rest of this session')

## Commands

Slash commands are answered by the agent, not by the terminal: there is one implementation of `/model`, and it is not in a frontend. `/help` is the only one this layer owns, because the keys it describes are this layer's.

In [ ]:
ui.buf.insert('/tools')
ui.submit()
[l for l in str(comp.blocks[max(comp.blocks)].body[0]).splitlines()[:4]]

['add_root', 'add_todo', 'create_file', 'create_skill']

In [ ]:
ui.buf.insert('/nope')
test_eq(ui.submit(), None)
comp.blocks[max(comp.blocks)].tag

'error'

A command is recognised before the options row. `/tools refactor` switches nothing on and runs the command. Otherwise any command with the word in it would open a menu instead.

In [ ]:
ui.buf.insert('/tools refactor')
test_eq(ui.submit(), None)
test_eq(ui.menu, None)                    # a command, never a question
comp.blocks[max(comp.blocks)].tag

'note'

An empty line does nothing at all, rather than sending an empty turn to the model.

In [ ]:
test_eq(ui.submit(), None)
ui.buf.text

''

## Running it

`mk_agent` is the assembly: a `LocalHost` over the folders named on the command line, an `Agent` over that, and the approval gate wired to both. `amain` is the tty loop, and it is short because everything it could get wrong lives in `Ui`.

In [ ]:
#| export
def mk_host(roots=('.',),
            approvals=None,          # an `Approvals` for the host to put writes in front of
            web=True,                # wire the web tools to fossick
            vault=False,             # keep what is read in a vishalakshi vault, for the next session
            spec=False,              # let the agent read an API specification and call it
            read_outside=False):     # let the read-only tools name any path on this machine
    "The host both frontends run on: `LocalHost`, plus a vault and an API spec when asked."
    bases = []
    if vault:
        from .vault import VaultHost
        bases.append(VaultHost)
    if spec:
        from .spec import SpecHost
        bases.append(SpecHost)
    Host = LocalHost if not bases else bases[0] if len(bases) == 1 else type('VaultSpecHost', tuple(bases), {})
    return Host(roots, approvals=approvals, web=web, read_outside=read_outside)


def mk_agent(roots=('.',),
             model=None,
             approve='ask',           # ask | auto | off | none (gate nothing at all)
             web=True,                # wire the web tools to fossick
             vault=False,             # keep what is read in a vishalakshi vault, for the next session
             spec=False,              # let the agent load OpenAPI/Azure/GCP specifications
             read_outside=False,      # let the read-only tools name any path on this machine
             **kw):                   # forwarded to `Agent`
    "A host over the named folders and an `Agent` over that, gated the way `approve` says."
    approvals = None if approve == 'none' else Approvals(tools=WRITE_TOOLS, mode=approve)
    host = mk_host(roots, approvals=approvals, web=web, vault=vault, spec=spec, read_outside=read_outside)
    if approvals is not None: approvals.host = host   # the gate previews `create_file` via the host
    agent = Agent(host, model=model, approvals=approvals, **kw)
    agent.lend_model()   # or a `--vault` session loads a second runtime for vishalakshi
    return agent, host

In [ ]:
#| export
async def amain(agent, hint='', python=False, attach=''):
    """The tty loop: one terminal, one event loop, one place that owns the keyboard.
    Bracketed paste on, mouse reporting off: the main screen is the terminal's to select from.
    `Ui.enter_transcript` borrows the mouse for the browsing view and gives it straight back, and
    `/mouse` lends it to the main screen for anyone who would rather click a block than select one.
    """
    tty = RealTty()
    tty.write('\x1b[?2004h')  # bracketed paste
    done = asyncio.Event()
    ui = None
    try:
        comp = await Compositor(tty).start()
        ui = Ui(comp, agent, loop=asyncio.get_running_loop())
        ui.hint = hint
        comp.on_task_error = lambda e, t: ui.say(Text(f'{t.get_name()} failed: {e!r}'), 'error')
        comp.spawn(ui.animate(), name='spinner')
        def on_key(k):
            out = ui.on_key(k)
            if out == 'quit': return done.set()
            if out is not None:
                ui.start_turn(out)
                ui.paint()
        comp.on_key = on_key
        comp.on_paste = ui.paste
        comp.on_resize = lambda: (comp.resize(), ui.paint())
        comp.on_mouse = ui.on_mouse
        ui.say(banner(agent.note, comp.console.width - 4), 'banner', fold=None)   # less the gutter
        if agent.plan:
            ui.say(plan_text(agent.plan.md()), 'plan', fold=None, source=agent.plan.md())
            ui.show_plan = True
        else: ui.say(Text('tab completes /commands · ctrl+t toggles the plan · /help for keys',
                        style=GRUVBOX['gray']), 'note', fold=None)
        if attach: ui.hint = f'{hint} · attached to {await ui.attach_session(attach)}'
        elif python: await ui.enter_python()
        ui.paint()
        loop = asyncio.get_running_loop()
        loop.add_reader(tty.fd, lambda: comp.on_bytes(tty.read(timeout=0)))
        try:
            while not done.is_set():          # the key parser needs periodic flushes (esc disambiguation)
                try: await asyncio.wait_for(done.wait(), 0.2)
                except asyncio.TimeoutError: comp.flush_input()
        finally:
            loop.remove_reader(tty.fd)
            comp.stop()
    finally:
        tty.write('\x1b[?2004l' + MOUSE_OFF + '\r\n')   # the view may have been up when this ended
        tty.restore()
        agent.close()
        if ui is not None and ui.kernel is not None: await ui.kernel.shutdown()

The agent that reaches the terminal is a real one over real folders, gated on writes:

In [ ]:
tempdir = tempfile.mkdtemp()
cli_agent, cli_host = mk_agent([tempdir], approve='ask', web=False)
cli_host.roots, sorted(WRITE_TOOLS & set(t.__name__ for t in cli_agent.tools))

(['/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpf02s57l2'],
 ['add_cell',
  'add_root',
  'create_file',
  'create_skill',
  'edit_cell',
  'edit_file',
  'git_checkout',
  'git_remote',
  'replace_text',
  'run_python',
  'run_shell'])

In [ ]:
test_eq(cli_agent.approvals.mode, 'ask')
test_eq(mk_agent([tempdir], approve='none')[0].approvals, None)
cli_agent.model.name

'gemma-e4b'

One turn with no terminal at all, for a pipe or a script. The entry point itself, which is `ramabana` on the command line.

In [ ]:
#| export
def ask_once(agent, prompt):
    "One turn with no terminal at all, for a pipe or a script. Returns the exit code."
    print(agent.ask(prompt))
    ok, problems = agent.ready, list(agent.problems)
    for p in problems: print(f'! {p}', file=sys.stderr)
    agent.close()
    return 0 if ok else 1

In [ ]:
#| export
@call_parse(pos=['prompt'])
def main(
    prompt: str = '',                    # one turn and exit. Omit for the interactive session
    root: str = '.',                     # folders it may read and write, comma separated: .,~/notes,/srv/app
    model: str = None,                   # the turn model. The routing default when omitted
    approve: str = 'ask',                # ask | auto | off | none (gate nothing at all)
    web: bool = True,                    # --no-web takes the network away from the web tools (fossick)
    read_outside: bool = False,          # widen reads to any path. Writing still needs the folder in --root
    subagent_writes: bool = False,       # let delegated sub-agents write, run commands and run Python too
    vault: bool = False,                 # vishalakshi vault for what is read. Not offered in python mode
    spec: bool = False,                 # enable OpenAPI, Azure and Google Discovery tools
    theme: str = 'auto',                 # a palette name; /theme in a session lists them all
    max_tool_calls: str = 'auto',         # auto | 20..400 tool calls per turn
    max_steps: str = 'auto',              # auto | 8..80 model/tool-loop steps per turn
    cfg: str = '~/.config/ramabana',     # config dir, for skills, extensions and resumable history
    resume: str = '',                    # saved session id/prefix, or 'latest'
    python: bool = False,                # start in python mode, on a kernel of your own
    attach: str = '',                  # join a live session; --kernels lists them
    kernels: bool = False,               # list live sessions and exit
):
    "Run Ramabana as a terminal agent or Python prompt. Name every folder it may work on: --root .,~/notes"
    if kernels:
        from ramabana.pyrepl import sessions
        print(sessions())
        return 0
    if vault and (python or attach):
        print('there is no vault-backed host for a dhrishti session; drop --vault', file=sys.stderr)
        return 2
    if attach:
        from ramabana.pyrepl import find_session
        try: attach = find_session(attach)
        except ModuleNotFoundError:
            print("--attach needs the pyrepl extra: pip install 'ramabana[pyrepl]'", file=sys.stderr)
            return 2
        except RuntimeError as e:   # `find_session` refuses in a sentence written for a human
            print(e, file=sys.stderr)
            return 2
    roots = [r.strip() for r in str(root).split(',') if r.strip()]
    try: set_theme(theme)
    except ValueError as e:
        print(e, file=sys.stderr)
        return 2
    try: agent, host = mk_agent(roots, model=model, approve=approve, web=web, vault=vault, spec=spec,
                               read_outside=read_outside, subagent_writes=subagent_writes,
                               max_tool_calls=max_tool_calls, max_steps=max_steps,
                               cfg=Path(cfg).expanduser() if cfg else None)
    except KeyError as e:
        print(e.args[0] if e.args else e, file=sys.stderr)   # `str` on a KeyError re-quotes it
        return 2
    except (RuntimeError, AgentError) as e:   # an unavailable runtime names the extra to install
        print(e, file=sys.stderr)
        return 2
    if resume:
        try: agent.resume_session(resume)
        except Exception as e:
            print(f'could not resume: {agent_err(e)}', file=sys.stderr)
            return 2
        if (was := getattr(agent, 'resumed_roots', None)):
            print(f'that session had also opened {", ".join(was)} · /root add PATH to open again',
                  file=sys.stderr)
    if agent.start() is None and not prompt:
        print(f'no model available: {agent.note}', file=sys.stderr)
    if prompt: return sys.exit(ask_once(agent, prompt))
    hint = f"{', '.join(host.roots)} · /python · /help"
    try: asyncio.run(amain(agent, hint, python=python, attach=attach))
    except KeyboardInterrupt: pass

In [ ]:
# One way in. Asserted at the parse layer, because calling `main(...)` in Python passes either way
# and proves nothing about argparse: `prompt` stays positional, and the mode is a flag.
from contextlib import redirect_stderr
from io import StringIO
from fastcore.script import anno_parser
_p = anno_parser(main.__wrapped__, pos=['prompt'])
test_eq(_p.parse_args(['one question', '--model', 'gpt']).prompt, 'one question')
test_eq(_p.parse_args([]).prompt, '')
test_eq(_p.parse_args(['--python']).python, True)
test_eq(_p.parse_args([]).python, False)
test_eq(_p.parse_args(['--attach', 'proj']).attach, 'proj')

# `--vault` swaps the host for one that keeps what it reads, which a dhrishti session cannot be:
# refused by name rather than silently dropped, and before anything is started.
_err = StringIO()
with redirect_stderr(_err): test_eq(main(python=True, vault=True), 2)
assert '--vault' in _err.getvalue()

# ...and an `--attach` that resolves to nothing is refused the same way, in the sentence
# `find_session` wrote and before a terminal or an agent exists.
import ramabana.pyrepl as _pyr
def _refuse(name): raise RuntimeError(f'no live dhrishti session matching {name!r}')
_real_find, _pyr.find_session = _pyr.find_session, _refuse
_err = StringIO()
try:
    with redirect_stderr(_err): test_eq(main(attach='nope'), 2)
finally: _pyr.find_session = _real_find
assert 'no live dhrishti session' in _err.getvalue()

`ask_once` prints the answer and reports anything that went wrong on stderr. `ramabana -p 'what changed?' | less` behaves like a Unix program.

In [ ]:
piped, _ = fake_agent(replies=['nothing changed since the last commit.'])
test_eq(ask_once(piped, 'what changed?'), 0)
piped.note

nothing changed since the last commit.


'fake · local · 1k ctx · modalities unknown · 26 tools'

The command line, as `--help` prints it:

In [ ]:
#| eval: false
!ramabana --help

usage: ramabana [-h] [--root ROOT] [--model MODEL] [--approve APPROVE]
                [--no-web] [--read_outside] [--subagent_writes] [--vault]
                [--spec] [--theme THEME] [--max_tool_calls MAX_TOOL_CALLS]
                [--max_steps MAX_STEPS] [--cfg CFG] [--resume RESUME] [--python]
                [--attach ATTACH] [--kernels]
                [prompt]

Run Ramabana as a terminal agent or Python prompt. Name every folder it may work
on: --root .,~/notes

positional arguments:
  prompt                           one turn and exit. Omit for the interactive
                                   session (default: )

options:
  -h, --help                       show this help message and exit
  --root ROOT                      folders it may read and write, comma
                                   separated: .,~/notes,/srv/app (default: .)
  --model MODEL                    the turn model. The routing default when
                                   omitted
  --approve APPROVE   

```
usage: ramabana [-h] [--prompt PROMPT] [--root ROOT] [--model MODEL]
                [--approve APPROVE] [--no-web] [--read_outside] [--vault]
                [--cfg CFG] [--resume RESUME] [--python] [--attach ATTACH]

Ramabana in a terminal: a coding agent over the folders you name, and a Python prompt in it.

  --prompt         run one turn and exit. Omit for the interactive session
  --root           folders the agent may touch, comma separated (default: .)
  --model          the turn model. The routing default when omitted
  --approve        ask | auto | off | none (gate nothing at all) (default: ask)
  --no-web         keep the web tools off the network
  --read_outside   let reads name any path on this machine. Writes stay inside
  --vault          keep what is read in a vishalakshi vault, for the next session
  --cfg            config dir, for skills, extensions and history
  --resume         saved session id/prefix, or 'latest'
  --python         start in python mode, on a kernel of your own
  --attach         a live dhrishti session by name or base URL. Agent only
```

So a session over this repository, on a local model, with nothing gated:

```bash
ramabana --root . --model qwen-4b --approve none
```

One that may read the rest of the machine but only write here:

```bash
ramabana --root . --read_outside
```

A Python prompt of your own, with the agent reading its namespace through Dhrishti's overlay:

```bash
ramabana --python --root .
```

And one turn for a pipe, with no terminal at all:

```bash
ramabana --prompt 'which files import fastllm?'
```

In [ ]:
# `submit` returns a coroutine for a turn, `'quit'` to leave, or None when it handled the line
# itself. Returning anything else reaches `comp.spawn`, which wants a coroutine.
ui.buf.insert('/guide'); test_eq(ui.submit(), None)
test_eq(ui.transcript.block_text(list(comp.blocks.values())[-1]), GUIDE)   # copy yields the text
test_eq(guide_text(GUIDE).plain.rstrip(), GUIDE.rstrip())                  # styling adds no text
assert guide_text(GUIDE).spans                                             # ...but does style it
assert max(len(l) for l in GUIDE.splitlines()) < 78                        # fits a narrow terminal

ui.buf.insert('/help');  test_eq(ui.submit(), None)
test_eq(ui.transcript.block_text(list(comp.blocks.values())[-1]), HELP)
assert key_card(HELP).spans
for q in ('/quit', '/exit', '/q'):
    ui.buf.insert(q); test_eq(ui.submit(), 'quit')

In [ ]:
#| export
@patch
def attach_file(self:Ui, path):
    "Attach one readable text/notebook file under the host's file policy."
    try: p = self.agent.host.check(path, must_exist=True, reading=True)
    except TypeError as e:
        if 'reading' not in str(e): return f'cannot attach {path}: {agent_err(e)}'
        try: p = self.agent.host.check(path, must_exist=True)
        except Exception as e: return f'cannot attach {path}: {agent_err(e)}'
    except Exception as e: return f'cannot attach {path}: {agent_err(e)}'
    p = Path(p).resolve()
    roots = [Path(r).resolve() for r in self.agent.host.roots]
    if not any(p == r or r in p.parents for r in roots): return f'cannot attach {p}: outside the open folders'
    if p.is_dir(): return f'cannot attach {p.name}: it is a folder'
    if p.suffix.lower() in MEDIA: return self.attach(p)
    text = self.agent.host.text_at(p)
    if text is None: return f'cannot attach {p.name}: not readable text'
    if len(text) > MAX_FILE_ATTACH: return f'cannot attach {p.name}: {_human(len(text))} is over the {_human(MAX_FILE_ATTACH)} limit'
    if len(self.attachments) >= MAX_ATTACH: return f'cannot attach {p.name}: {MAX_ATTACH} is the limit'
    a = FileAttachment(p, text)
    if any(x.path == a.path for x in self.attachments): return f'{a.name} is already attached'
    self.attachments.append(a)
    return f'attached file {a.label()}'

@patch
def _file_matches(self:Ui):
    "Root-scoped completion candidates for the final `@query` in the prompt."
    m = re.search(r'(?<!\S)@([^\s]*)$', self.buf.text[:self.buf.cursor])
    if m is None: return None
    query = m.group(1).lower()
    roots = [Path(r) for r in self.agent.host.roots]
    rows = []
    for p in map(Path, self.agent.host.walk()):
        try:
            root = next(r for r in roots if p == r or r in p.parents)
            rel = str(p.relative_to(root))
        except (StopIteration, ValueError): continue
        if query in rel.lower(): rows.append(rel)
    return (m.start(1), sorted(rows))


@patch
def _refresh_complete(self:Ui):
    "Refresh slash-command or root-scoped `@` file completion."
    hit = self._file_matches()
    if hit is not None:
        start, matches = hit
        self.complete = CompletionMenu(self.buf, matches, start=start, show=8) if matches else None
        return
    hits = self._slash_matches()
    self.complete = CompletionMenu(self.buf, hits, start=0, show=8) if hits else None


_core_on_key_files = Ui.on_key
@patch
def on_key(self:Ui, k):
    "Offer `@` completion with the same Tab menu used for slash commands."
    if self.ask is None and k.name == 'tab' and self.complete is None and self._file_matches() is not None:
        self._refresh_complete()
        if self.complete is not None:
            if not self.complete.insert_common(): self.complete.cycle(1)
            return self.paint()
    out = _core_on_key_files(self, k)
    if self.ask is None and self.complete is None and self._file_matches() is not None: self._refresh_complete()
    return out


_core_run_turn_files = run_turn
async def run_turn(ui, prompt):
    "Run a turn after capturing its readable `@` file references."
    for ref in file_refs(prompt): ui.attach_file(ref)
    return await _core_run_turn_files(ui, prompt)

In [ ]:
#| export
@patch
def theme_list(self:Ui):
    "The palettes `/theme NAME` will take, the one on screen in brackets."
    names = ' · '.join(f'[{n}]' if n == ACTIVE_THEME else n for n in THEMES)
    return self.note(f'themes: {names} · /theme NAME or /theme next')


@patch
def apply_theme(self:Ui, name=''):
    "Apply a terminal palette and repaint this UI. `next` and `prev` step along `THEMES`."
    names = list(THEMES)
    if name in ('next', 'prev'):
        step = 1 if name == 'next' else -1
        name = names[(names.index(ACTIVE_THEME) + step) % len(names)]
    try: active = set_theme(name or ACTIVE_THEME)
    except ValueError as e: return self.note(str(e), 'error')
    self.theme = active
    self.comp.console.push_theme(MARKDOWN_THEME)
    self.paint()
    # A block keeps the `Text` it was printed as, so the transcript above stays in the palette it
    # was painted in. Say so, rather than leave a reader wondering why half the screen did not move.
    return self.note(f'theme: {active} · the bar and what comes next')


@patch
def open_root(self:Ui, path=''):
    """`/root` lists the open folders; `/root add PATH` opens another.
    Typed by the person whose machine it is, so it is not gated: they are the approval. The agent
    asking for the same thing goes through `add_root`, which is in `WRITE_TOOLS`.
    """
    host = self.agent.host
    if not path:
        added = set(getattr(host, 'added_roots', []))
        rows = [f'{r}  (opened this session)' if r in added else r for r in host.roots]
        return self.note('open folders: ' + ' · '.join(rows))
    bits = path.split(None, 1)
    if bits[0] != 'add' or len(bits) != 2: return self.note('usage: /root [add PATH]', 'error')
    try: opened = host.add_root(bits[1].strip())
    except Exception as e: return self.note(agent_err(e), 'error')
    return self.note(f'opened {opened} · reads and writes now reach it')


_submit_theme = Ui.submit
@patch
def submit(self:Ui):
    "Handle the UI-owned `/theme` and `/root` commands before ordinary terminal commands."
    line = self.buf.text.strip()
    if line == '/root' or line.startswith('/root '):
        self.buf.clear()
        return self.open_root(line[len('/root'):].strip())
    if line == '/theme' or line.startswith('/theme '):
        self.buf.clear()
        bits = line.split()
        if len(bits) == 1: return self.theme_list()
        if len(bits) == 2: return self.apply_theme(bits[1])
        return self.note('usage: /theme [NAME|next|prev]', 'error')
    return _submit_theme(self)

In [ ]:
#| export
@patch
def stop(self:Ui):
    "Escalate repeated Ctrl-C from cancellation to termination to exit."
    now = time.monotonic()
    if now - self._stop_at > self.agent.cancel_grace:
        self._stop_count, self._stop_run = 0, ''
    self._stop_at, self._stop_count = now, self._stop_count + 1
    if self._stop_count >= 3:return 'quit'
    if self.turn is not None:
        self.flush_stream()   # keep what it managed to say...
        self._seg_blk = None  # ...but nothing later may grow it from above the note
    run = self.agent.run(self._stop_run) if self._stop_run else self.agent.run()
    if run is None:
        self._stop_count, self._stop_run = 0, ''
        return self.note('idle')
    self._stop_run = run.id
    if self._stop_count == 2:
        self.note('terminating')
        fn = lambda: self.agent.terminate(run.id)
    else:
        self.note('cancelling')
        fn = lambda: self.agent.cancel(run.id)
    def work():
        state = fn()['state']
        self._post(self.note, state)
    threading.Thread(target=work, daemon=True).start()
    return self.paint()


In [ ]:
#| export
_ui_on_key = Ui.on_key

@patch
def on_key(self:Ui, k):
    if (k.name == 'ctrl+c' and self.mode != 'python' and self.ask is None
            and (self.turn is not None or self._stop_count)):
        self.buf.clear()
        return self.stop()
    return _ui_on_key(self, k)


In [ ]:
class _StopAgent:
    cancel_grace = 1
    def __init__(self):
        self.r = type('R', (), {'id': 'r1', 'terminal': False})()
    def run(self, run_id=''): return self.r
    def cancel(self, run_id): self.r.terminal = True; return {'state': 'detached'}
    def terminate(self, run_id): return {'state': 'terminated'}

class _StopUi:
    def __init__(self):
        self.agent, self._stop_at, self._stop_count, self._stop_run = _StopAgent(), 0., 0, ''
        self.notes, self.turn = [], None   # no transcript here: the ladder is what is under test
    def note(self, text): self.notes.append(text)
    def paint(self): return None
    def _post(self, fn, *a): return fn(*a)

u = _StopUi()
Ui.stop(u); time.sleep(.02)
Ui.stop(u); time.sleep(.02)
test_eq(u.notes[:4], ['cancelling', 'detached', 'terminating', 'terminated'])
test_eq(Ui.stop(u), 'quit')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
from inspect import signature

assert 'graph' not in signature(mk_host).parameters
assert 'graph' not in signature(mk_agent).parameters
assert 'graph' not in signature(main).parameters

In [ ]:
tl_ui.agent.activity.since()[-ACT_TAIL:]

[Act(tool='search_code', args={'query': 'threshold callers'}, summary='Search threshold callers', detail='hits', ok=True, done=True, secs=0.0, started=1787183168.097574, id='04f48ae3fc71', turn_id='', revision=0, branch_id='main', parent_action_id='57d71d0683b8', state='complete'),
 Act(tool='search_code', args={'query': 'reserve usage'}, summary='Search reserve usage', detail='hits', ok=True, done=True, secs=0.0, started=1787183168.098463, id='e3d50cd275bb', turn_id='', revision=0, branch_id='main', parent_action_id='57d71d0683b8', state='complete'),
 Act(tool='run_shell', args={'command': 'pytest'}, summary='Run pytest', detail='FAILED test_reserve\nassert 0.9 <= 0.8', ok=False, done=True, secs=0.0, started=1787183168.100472, id='6081419092eb', turn_id='', revision=0, branch_id='main', parent_action_id='', state='failed')]

In [ ]:
tl_ui.turn = 'a turn'
footer = [r.plain for r in tl_ui.working()]
assert all(row[:8].count(':') != 2 for row in footer), footer
assert not any('─' * 18 in row for row in footer), footer